# Satellite Methane Leak Intelligence
### An end-to-end environmental data engineering project

**Goal:** turn published satellite methane plume observations into a reproducible warehouse, quality report, and geographic investigation queue.

**Stack:** Python · HTTP API · Parquet · DuckDB SQL · H3 · Folium · Plotly · pytest.

**One-file start:** upload this notebook to [Google Colab](https://colab.research.google.com/), use a standard CPU runtime, and choose **Runtime → Run all**. No GPU or separate dataset upload is needed for the default live run. The notebook writes its own project source files and finishes with a downloadable project/results ZIP.

The default scope is global, published CH4 observations from Tanager and EMIT, starting 2024-01-01. A verified build on 2026-10-25 collected 18,951 observations; a later live run may differ. The first collection took several minutes in validation. Requests are paginated, paced, and retried where appropriate.

**What the project measures:** published plume observations and repeated detection dates in geographic grid cells. It uses processed Carbon Mapper data. A grid cell is not a confirmed facility; a reported instantaneous rate is not an annual emissions total. Missing quality ratings stay visible throughout the analysis.

**Outputs and run history:** this notebook starts without saved code outputs. Running it creates ordinary notebook outputs, pipeline JSONL logs, and audit tables. Google controls Colab's own history and service records. [Colab sharing and runtime documentation](https://research.google.com/colaboratory/faq.html).

Source: Carbon Mapper. [Product guide](https://carbonmapper.org/articles/product-guide) · [API reference](https://api.carbonmapper.org/api/v1/docs) · [Data terms](https://carbonmapper.org/terms). Source data and derivatives retain the provider's non-commercial use and attribution conditions.

## 1. Environment and settings

The standard settings run without Drive access. Enabling `USE_GOOGLE_DRIVE` adds Google's authorization prompt and saves a checkpoint at the end. Local Colab files are temporary, so the final download matters when Drive is disabled.

`BBOX = None` means global coverage. `(-125, 32, -114, 42)` is a smaller regional rectangle for a quicker run. Scope changes get a separate warehouse automatically.

In [ ]:
import importlib.metadata
import json
import os
import subprocess
import sys
from pathlib import Path

# These are the package versions I used for the pipeline; an existing pandas 2.x is fine.
PACKAGES = ["pandas>=2.2,<3", "duckdb==1.4.3", "pyarrow==19.0.1", "requests==2.32.5",
            "folium==0.20.0", "plotly==6.3.0", "h3==4.3.1", "pytest==8.4.2", "filelock==3.20.0"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", *PACKAGES], check=True)

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# I can enable Drive when I want this warehouse to survive a new Colab runtime.
USE_GOOGLE_DRIVE = False
START_DATE = "2024-01-01"
END_DATE = None
BBOX = None
MODE = "auto"
RUN_INCREMENTAL_CHECK = True
FETCH_ONE_PLUME_IMAGE = True
AUTO_DOWNLOAD_ZIP = True

# A completed manifest from the accompanying bundle also lets me run without the catalog API.
OFFLINE_MANIFEST = None

default_project = "/content/methane_project" if IN_COLAB else str(Path.cwd() / "methane_project")
PROJECT_DIR = Path(os.environ.get("METHANE_PROJECT_DIR", default_project)).resolve()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
for folder in ["tests", "docs", "sql", "verification", ".github/workflows"]:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Project folder:", PROJECT_DIR)
print("Runtime:", sys.version.split()[0], "| Colab:", IN_COLAB)

## 2. Reusable pipeline source

The following cells write a real Python module. Keeping the same functions in a file lets me use them from this notebook, tests, a command-line job, or a container. The cells are split by responsibility so I can return to a particular part later.

The data flow is **API → archived JSON → validated Parquet → transactional DuckDB → SQL marts → report and exports**. Run state and revision history make the flow repeatable.

### 2.1. Configuration and small helpers

Dataset scope and runtime settings are separate. A scope fingerprint keeps unrelated selections in separate warehouses.

In [ ]:
%%writefile methane_pipeline.py
"""Satellite Methane Leak Intelligence: a reproducible local batch pipeline."""

# %% Configuration and small helpers
from __future__ import annotations

import argparse
import dataclasses
import gzip
import hashlib
import html
import json
import math
import os
import shutil
import sys
import time
import uuid
from contextlib import contextmanager
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

import duckdb
from filelock import FileLock, Timeout as FileLockTimeout
import h3
import pandas as pd
import requests

API = "https://api.carbonmapper.org/api/v1/catalog/plumes/annotated"
CONTRACT_VERSION = "1"
SECTORS = {
    "1A1": "Energy industries", "1A2": "Manufacturing industries",
    "1B1a": "Coal mining", "1B2": "Oil and natural gas",
    "4A": "Enteric fermentation", "4B": "Manure management",
    "6A": "Solid waste disposal", "6B": "Wastewater handling",
    "other": "Other", "unknown": "Undetermined",
}


def utcnow():
    return datetime.now(timezone.utc)


def iso(value):
    return value.isoformat().replace("+00:00", "Z")


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),
                                     ensure_ascii=False, allow_nan=False).encode()).hexdigest()


def write_json(path, value):
    # A temporary file keeps an interrupted write from replacing a good checkpoint.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    os.replace(tmp, path)


@dataclass
class Config:
    root: str = "runtime"
    start_date: str = "2024-01-01"
    end_date: str | None = None
    bbox: tuple | None = None
    instruments: tuple = ("tan", "emi")
    page_size: int = 1000
    max_pages: int = 250
    overlap_hours: int = 48
    full_refresh_days: int = 7
    h3_resolution: int = 8
    request_pause: float = 0.35
    timeout_seconds: int = 60
    max_retries: int = 4
    max_map_cells: int = 2000

    def __post_init__(self):
        self.root = str(Path(self.root).resolve())
        self.instruments = tuple(self.instruments)
        if not self.instruments or not set(self.instruments) <= {"tan", "emi"}:
            raise ValueError("This project is scoped to satellite instruments tan and emi.")
        pd.Timestamp(self.start_date, tz="UTC")
        if self.end_date is not None:
            if pd.Timestamp(self.end_date, tz="UTC") <= pd.Timestamp(self.start_date, tz="UTC"):
                raise ValueError("end_date must be later than start_date.")
        if self.bbox is not None:
            self.bbox = tuple(float(x) for x in self.bbox)
            if len(self.bbox) != 4:
                raise ValueError("bbox needs west, south, east, north.")
            west, south, east, north = self.bbox
            if not (-180 <= west < east <= 180 and -90 <= south < north <= 90):
                raise ValueError("Invalid bounding box; antimeridian boxes need separate runs.")
        if not 1 <= self.page_size <= 1000 or self.max_pages < 1:
            raise ValueError("Use page_size 1..1000 and a positive max_pages.")
        if not 0 <= self.h3_resolution <= 15 or self.overlap_hours < 0:
            raise ValueError("Invalid H3 resolution or overlap.")
        if self.max_retries < 0 or self.timeout_seconds <= 0 or self.request_pause < 0:
            raise ValueError("Retry, timeout, and pause settings must be non-negative, with a positive timeout.")
        if self.full_refresh_days < 1 or self.max_map_cells < 1:
            raise ValueError("full_refresh_days and max_map_cells must be positive.")

    @property
    def scope(self):
        # Transport settings do not change the scientific dataset I am collecting.
        return {"start_date": self.start_date, "end_date": self.end_date,
                "bbox": self.bbox, "instruments": sorted(self.instruments),
                "gas": "CH4", "status": "published", "contract": CONTRACT_VERSION,
                "h3_resolution": self.h3_resolution}

    @property
    def scope_id(self):
        return digest(self.scope)[:16]

    @property
    def work(self):
        return Path(self.root) / self.scope_id

    @property
    def db_path(self):
        return self.work / "warehouse.duckdb"


class RunLog:
    def __init__(self, path, run_id):
        self.path, self.run_id = Path(path), run_id
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def event(self, event, **values):
        row = {"at_utc": iso(utcnow()), "run_id": self.run_id, "event": event, **values}
        with self.path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(row, default=str) + "\n")
        print(f"[{event}] " + " ".join(f"{k}={v}" for k, v in values.items()), flush=True)


@contextmanager
def project_lock(work):
    # I only want one process changing this warehouse at a time.
    work = Path(work)
    work.mkdir(parents=True, exist_ok=True)
    lock = work / ".pipeline.lock"
    try:
        # The operating system releases this lock if a notebook process is interrupted.
        with FileLock(str(lock), timeout=0):
            yield
    except FileLockTimeout as exc:
        raise RuntimeError("Another process holds the warehouse lock. Wait for that run to finish "
                           "or stop it before starting this run.") from exc

### 2.2. HTTP collection, raw archives, and verified replay

Every raw page has a checksum and a manifest entry. The filtered result count drives pagination. A partial or inconsistent extraction never reaches the warehouse.

In [ ]:
%%writefile -a methane_pipeline.py
# %% HTTP collection, raw archives, and verified replay
def fetch_json(session, params, cfg, log, sleep=time.sleep):
    # Temporary failures should get another chance; authentication errors should be visible.
    for attempt in range(cfg.max_retries + 1):
        try:
            response = session.get(API, params=params, timeout=cfg.timeout_seconds)
            if response.status_code in (401, 403):
                raise PermissionError("Carbon Mapper denied access. Check its current API access "
                                      "requirements; no fallback data has been substituted.")
            retryable = response.status_code == 429 or response.status_code >= 500
            if retryable:
                if attempt == cfg.max_retries:
                    response.raise_for_status()
                wait = min(30.0, 2.0 ** attempt)
                try:
                    wait = min(60.0, max(wait, float(response.headers.get("Retry-After", 0))))
                except ValueError:
                    pass
                log.event("retry", attempt=attempt + 1, status=response.status_code, wait_seconds=wait)
                sleep(wait)
                continue
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, dict) or not isinstance(payload.get("items"), list):
                raise ValueError("API contract changed: expected an object containing an items list.")
            return payload
        except (requests.ConnectionError, requests.Timeout) as exc:
            if attempt == cfg.max_retries:
                raise
            wait = min(30.0, 2.0 ** attempt)
            log.event("retry", attempt=attempt + 1, reason=type(exc).__name__, wait_seconds=wait)
            sleep(wait)
    raise RuntimeError("Retry loop ended unexpectedly.")


def make_query(cfg, cutoff, lower_modified=None):
    query = {"plume_gas": "CH4", "status": "published", "instruments": list(cfg.instruments),
             "sort": "asc", "limit": cfg.page_size,
             "datetime": f"{cfg.start_date}T00:00:00Z/" + (
                 f"{cfg.end_date}T00:00:00Z" if cfg.end_date else cutoff),
             "modified_at": f"{lower_modified or '..'}/{cutoff}"}
    if cfg.bbox is not None:
        query["bbox"] = list(cfg.bbox)
    return query


def extract(cfg, run_id, cutoff, mode, lower_modified, log, session=None):
    directory = cfg.work / "bronze" / run_id
    directory.mkdir(parents=True, exist_ok=False)
    manifest = {"run_id": run_id, "scope_id": cfg.scope_id, "scope": cfg.scope,
                "mode": mode, "cutoff_utc": cutoff, "lower_modified_utc": lower_modified,
                "endpoint": API, "query": make_query(cfg, cutoff, lower_modified),
                "complete": False, "pages": [], "source": "Carbon Mapper",
                "terms": "https://carbonmapper.org/terms"}
    manifest_path = directory / "manifest.json"
    write_json(manifest_path, manifest)
    own_session = session is None
    session = session or requests.Session()
    if own_session:
        session.headers["User-Agent"] = "MethaneObservationPipeline/1.0 (research portfolio)"
    offset, expected, seen = 0, None, set()
    try:
        for page_index in range(cfg.max_pages):
            query = {**manifest["query"], "offset": offset}
            payload = fetch_json(session, query, cfg, log)
            # bbox_count is the filtered count inside the requested box; total_count can be larger.
            count = payload.get("bbox_count", payload.get("total_count"))
            if not isinstance(count, int) or count < 0:
                raise ValueError("API contract changed: missing integer result count.")
            if expected is None:
                expected = count
            if count != expected:
                raise RuntimeError("The source changed while paging. Raw pages are retained; "
                                   "rerun to start a new consistent extraction.")
            items = payload["items"]
            if not items and offset < expected:
                raise RuntimeError("An empty page arrived before all matching records were collected.")
            page_ids = [x.get("id") if isinstance(x, dict) else None for x in items]
            if any(not isinstance(i, str) or not i for i in page_ids):
                raise ValueError("Missing observation IDs in API response.")
            if len(set(page_ids)) != len(page_ids) or seen.intersection(page_ids):
                raise RuntimeError("Duplicate IDs across API pages; refusing an incomplete snapshot.")
            seen.update(page_ids)
            encoded = json.dumps(payload, ensure_ascii=False, separators=(",", ":"), allow_nan=False).encode()
            name = f"page_{page_index:05d}.json.gz"
            with gzip.open(directory / name, "wb") as f:
                f.write(encoded)
            manifest["pages"].append({"file": name, "offset": offset, "rows": len(items),
                                      "sha256": hashlib.sha256(encoded).hexdigest()})
            offset += len(items)
            manifest.update(expected_rows=expected, rows_archived=offset, next_offset=offset)
            write_json(manifest_path, manifest)
            log.event("page_saved", page=page_index + 1, rows=len(items), collected=offset, expected=expected)
            if offset == expected:
                manifest["complete"] = True
                manifest["finished_at_utc"] = iso(utcnow())
                write_json(manifest_path, manifest)
                return manifest_path
            if offset > expected:
                raise RuntimeError("Received more rows than the API's filtered count.")
            time.sleep(cfg.request_pause)
        raise RuntimeError("max_pages reached. No warehouse changes were made; increase the limit explicitly.")
    finally:
        if own_session:
            session.close()


def read_archive(manifest_path, scope_id=None):
    path = Path(manifest_path)
    manifest = json.loads(path.read_text())
    if not manifest.get("complete"):
        raise ValueError("This archive is incomplete; it cannot be loaded into the warehouse.")
    if scope_id and manifest["scope_id"] != scope_id:
        raise ValueError("Archive and project use different dataset scopes.")
    records, seen = [], set()
    for page in manifest["pages"]:
        page_path = (path.parent / page["file"]).resolve()
        if page_path.parent != path.parent.resolve():
            raise ValueError("Unsafe archive page path.")
        with gzip.open(page_path, "rb") as f:
            body = f.read()
        if hashlib.sha256(body).hexdigest() != page["sha256"]:
            raise ValueError(f"Raw-page checksum mismatch: {page['file']}")
        items = json.loads(body)["items"]
        if len(items) != page["rows"]:
            raise ValueError("Raw page row count does not match its manifest.")
        for item in items:
            if item.get("id") in seen:
                raise ValueError("Duplicate ID in saved archive.")
            seen.add(item.get("id"))
            records.append((item, f"bronze/{manifest['run_id']}/{page['file']}"))
    if len(records) != manifest["expected_rows"]:
        raise ValueError("Archive is missing expected records.")
    return manifest, records

### 2.3. Normalization and scientific data checks

GeoJSON coordinates arrive as longitude, latitude; H3 expects latitude, longitude. Provider-hidden estimates are masked. Missing optional values remain null, and critical failures stop the batch.

In [ ]:
%%writefile -a methane_pipeline.py
# %% Normalization and scientific data checks
def stable_value(value):
    # Expiring image signatures are transport details, not new scientific observations.
    if isinstance(value, dict):
        return {k: stable_value(v) for k, v in value.items()}
    if isinstance(value, list):
        return [stable_value(v) for v in value]
    if isinstance(value, str) and value.startswith("https://"):
        parts = urlsplit(value)
        keep = [(k, v) for k, v in parse_qsl(parts.query, keep_blank_values=True)
                if k.lower() not in {"expires", "signature", "key-pair-id"}
                and not k.lower().startswith("x-amz-")]
        return urlunsplit((parts.scheme, parts.netloc, parts.path, urlencode(keep), parts.fragment))
    return value


def number(value):
    if value is None or isinstance(value, bool):
        return None
    try:
        val = float(value)
        return val if math.isfinite(val) else None
    except (TypeError, ValueError):
        return None


def timestamp(value):
    # A scalar Timestamp avoids repeating pandas' general date-inference work for every field.
    if not isinstance(value, (str, datetime, pd.Timestamp)):
        return None
    try:
        val = pd.Timestamp(value)
        if pd.isna(val):
            return None
        return val.tz_localize("UTC") if val.tzinfo is None else val.tz_convert("UTC")
    except (TypeError, ValueError):
        return None


DATA_TYPES = {
    "observation_id": "VARCHAR", "plume_id": "VARCHAR", "gas": "VARCHAR",
    "scene_id": "VARCHAR", "observed_at": "TIMESTAMPTZ", "modified_at": "TIMESTAMPTZ",
    "published_at": "TIMESTAMPTZ", "instrument": "VARCHAR", "platform": "VARCHAR",
    "sector_code": "VARCHAR", "longitude": "DOUBLE", "latitude": "DOUBLE",
    "emission_rate_kg_hr": "DOUBLE", "uncertainty_kg_hr": "DOUBLE",
    "wind_speed_m_s": "DOUBLE", "wind_direction_deg": "DOUBLE",
    "plume_quality": "VARCHAR", "hide_emission": "BOOLEAN", "review_eligible": "BOOLEAN",
    "processing_version": "VARCHAR", "h3_cell": "VARCHAR", "quality_flags": "VARCHAR",
    "record_hash": "VARCHAR", "raw_page": "VARCHAR",
}


def normalize(records, cfg):
    rows, rejected = [], []
    for item, raw_page in records:
        errors, flags = [], []
        oid, name = item.get("id"), item.get("plume_id")
        if not isinstance(oid, str) or not oid or not isinstance(name, str) or not name:
            errors.append("missing_id")
        if item.get("gas") != "CH4":
            errors.append("wrong_gas")
        if item.get("instrument") not in cfg.instruments:
            errors.append("wrong_instrument")
        if item.get("status") != "published":
            errors.append("not_published")
        geometry = item.get("geometry_json")
        if not isinstance(geometry, dict):
            geometry = {}
        if geometry.get("type") != "Point":
            errors.append("invalid_geometry_type")
        coords = geometry.get("coordinates", [])
        lon = number(coords[0]) if isinstance(coords, list) and len(coords) >= 2 else None
        lat = number(coords[1]) if isinstance(coords, list) and len(coords) >= 2 else None
        if lon is None or lat is None or not (-180 <= lon <= 180 and -90 <= lat <= 90):
            errors.append("invalid_coordinates")
        elif cfg.bbox:
            w, s, e, n = cfg.bbox
            if not (w <= lon <= e and s <= lat <= n):
                errors.append("outside_requested_box")
        observed, modified = timestamp(item.get("scene_timestamp")), timestamp(item.get("modified"))
        if observed is None or modified is None:
            errors.append("invalid_required_timestamp")
        elif observed < pd.Timestamp(cfg.start_date, tz="UTC"):
            errors.append("before_requested_period")
        elif cfg.end_date and observed > pd.Timestamp(cfg.end_date, tz="UTC"):
            errors.append("after_requested_period")
        if not item.get("scene_id"):
            errors.append("missing_scene_id")
        if not isinstance(item.get("hide_emission"), bool):
            errors.append("missing_visibility_flag")
        if errors:
            rejected.append({"observation_id": oid, "reasons": errors, "raw_page": raw_page})
            continue
        hidden = item["hide_emission"]
        rate, uncertainty = number(item.get("emission_auto")), number(item.get("emission_uncertainty_auto"))
        if rate is not None and rate < 0:
            rate = None
            flags.append("invalid_negative_rate")
        if uncertainty is not None and uncertainty < 0:
            uncertainty = None
            flags.append("invalid_negative_uncertainty")
        if hidden:
            rate, uncertainty = None, None
            flags.append("estimate_withheld_by_provider")
        quality = item.get("plume_quality") or "not_supplied"
        if quality not in {"good", "questionable", "bad", "not_supplied"}:
            flags.append("unrecognized_quality_rating")
        if quality == "not_supplied":
            flags.append("quality_not_supplied")
        if rate is None:
            flags.append("rate_unavailable")
        if uncertainty is None:
            flags.append("uncertainty_unavailable")
        sector = item.get("sector") or "unknown"
        if sector not in SECTORS:
            flags.append("unmapped_sector")
        rows.append({
            "observation_id": oid, "plume_id": name, "gas": "CH4",
            "scene_id": str(item["scene_id"]), "observed_at": observed, "modified_at": modified,
            "published_at": timestamp(item.get("published_at")),
            "instrument": item["instrument"], "platform": str(item.get("platform") or "Unknown"),
            "sector_code": sector, "longitude": lon, "latitude": lat,
            "emission_rate_kg_hr": rate, "uncertainty_kg_hr": uncertainty,
            "wind_speed_m_s": number(item.get("wind_speed_avg_auto")),
            "wind_direction_deg": number(item.get("wind_direction_avg_auto")),
            "plume_quality": quality, "hide_emission": hidden,
            "review_eligible": rate is not None and rate > 0 and quality in {"good", "not_supplied"},
            "processing_version": str(item.get("emission_version") or "unknown"),
            "h3_cell": h3.latlng_to_cell(lat, lon, cfg.h3_resolution),
            "quality_flags": ";".join(flags), "record_hash": digest(stable_value(item)),
            "raw_page": raw_page,
        })
    frame = pd.DataFrame(rows, columns=DATA_TYPES)
    for col, dtype in DATA_TYPES.items():
        if dtype == "TIMESTAMPTZ":
            frame[col] = pd.to_datetime(frame[col], utc=True)
        elif dtype == "DOUBLE":
            frame[col] = pd.to_numeric(frame[col], errors="coerce").astype("float64")
        elif dtype == "BOOLEAN":
            frame[col] = frame[col].astype("bool")
        else:
            frame[col] = frame[col].astype("string")
    if frame.observation_id.duplicated().any():
        raise ValueError("Duplicate observation IDs after normalization.")
    checks = [
        {"check": "critical_contract", "severity": "error", "failures": len(rejected)},
        {"check": "quality_rating_not_supplied", "severity": "warning",
         "failures": int((frame.plume_quality == "not_supplied").sum())},
        {"check": "rate_unavailable", "severity": "warning", "failures": int(frame.emission_rate_kg_hr.isna().sum())},
        {"check": "uncertainty_unavailable", "severity": "warning", "failures": int(frame.uncertainty_kg_hr.isna().sum())},
    ]
    return frame, rejected, checks

### 2.4. Transactional warehouse and SQL models

Observation IDs are the merge keys. Canonical hashes distinguish revisions, while provider modification time prevents an older replay from replacing a newer observation. Full refreshes also reconcile withdrawals.

In [ ]:
%%writefile -a methane_pipeline.py
# %% Transactional warehouse and SQL models
def initialize_db(con):
    con.execute("SET TimeZone='UTC'")
    definitions = ", ".join(f'"{name}" {kind}' for name, kind in DATA_TYPES.items())
    con.execute(f"CREATE TABLE IF NOT EXISTS fact_observations ({definitions}, PRIMARY KEY(observation_id))")
    con.execute(f"CREATE TABLE IF NOT EXISTS observation_versions ({definitions}, "
                "first_seen_run VARCHAR, first_seen_at TIMESTAMPTZ, PRIMARY KEY(observation_id, record_hash))")
    con.execute("CREATE TABLE IF NOT EXISTS pipeline_runs (run_id VARCHAR PRIMARY KEY, mode VARCHAR, "
                "started_at TIMESTAMPTZ, finished_at TIMESTAMPTZ, status VARCHAR, metrics_json VARCHAR, error VARCHAR)")
    con.execute("CREATE TABLE IF NOT EXISTS quality_results (run_id VARCHAR, check_name VARCHAR, severity VARCHAR, failures BIGINT)")
    con.execute("CREATE TABLE IF NOT EXISTS retirements (run_id VARCHAR, observation_id VARCHAR, retired_at TIMESTAMPTZ)")


def load_current(con, frame, run_id, cutoff, full_refresh):
    con.register("incoming_frame", frame)
    casts = ", ".join(f'CAST("{name}" AS {kind}) AS "{name}"' for name, kind in DATA_TYPES.items())
    con.execute(f"CREATE OR REPLACE TEMP TABLE incoming AS SELECT {casts} FROM incoming_frame")
    new_rows = con.execute("SELECT count(*) FROM incoming i LEFT JOIN fact_observations f "
                           "USING(observation_id) WHERE f.observation_id IS NULL").fetchone()[0]
    # An older replay should never replace a more recent provider revision.
    con.execute("""CREATE OR REPLACE TEMP TABLE changes AS
        SELECT i.* FROM incoming i LEFT JOIN fact_observations f USING(observation_id)
        WHERE f.observation_id IS NULL OR
              (i.modified_at >= f.modified_at AND i.record_hash <> f.record_hash)""")
    changes = con.execute("SELECT count(*) FROM changes").fetchone()[0]
    con.execute("""INSERT INTO observation_versions
        SELECT i.*, ?, CAST(? AS TIMESTAMPTZ) FROM incoming i
        WHERE NOT EXISTS (SELECT 1 FROM observation_versions v
            WHERE v.observation_id=i.observation_id AND v.record_hash=i.record_hash)""", [run_id, cutoff])
    con.execute("DELETE FROM fact_observations WHERE observation_id IN (SELECT observation_id FROM changes)")
    con.execute("INSERT INTO fact_observations SELECT * FROM changes")
    removed = 0
    if full_refresh:
        # A complete refresh also reconciles observations withdrawn from the published catalog.
        removed = con.execute("SELECT count(*) FROM fact_observations WHERE observation_id NOT IN "
                              "(SELECT observation_id FROM incoming)").fetchone()[0]
        con.execute("INSERT INTO retirements SELECT ?, observation_id, CAST(? AS TIMESTAMPTZ) "
                    "FROM fact_observations WHERE observation_id NOT IN (SELECT observation_id FROM incoming)",
                    [run_id, cutoff])
        con.execute("DELETE FROM fact_observations WHERE observation_id NOT IN (SELECT observation_id FROM incoming)")
    con.unregister("incoming_frame")
    return {"new": new_rows, "updated": changes - new_rows,
            "unchanged_or_older": len(frame) - changes, "removed": removed}


MODELS_SQL = """
CREATE OR REPLACE TABLE dim_platform AS
SELECT DISTINCT instrument, platform FROM fact_observations;

CREATE OR REPLACE TABLE dim_sector AS
SELECT DISTINCT f.sector_code, coalesce(s.sector_name, 'Unmapped: ' || f.sector_code) AS sector_name
FROM fact_observations f LEFT JOIN sector_lookup s USING (sector_code);

CREATE OR REPLACE TABLE dim_date AS
SELECT DISTINCT CAST(observed_at AS DATE) AS observation_date,
       year(observed_at)::INTEGER AS year, month(observed_at)::INTEGER AS month,
       dayofweek(observed_at)::INTEGER AS weekday
FROM fact_observations;

CREATE OR REPLACE TABLE mart_monthly AS
SELECT date_trunc('month', observed_at)::DATE AS observation_month,
       instrument, count(*) AS observations, count(DISTINCT scene_id) AS scenes,
       count(DISTINCT h3_cell) AS occupied_cells,
       count(*) FILTER (WHERE review_eligible) AS review_eligible_observations
FROM fact_observations GROUP BY 1, 2;

CREATE OR REPLACE TABLE mart_sectors AS
SELECT f.sector_code, s.sector_name, count(*) AS observations,
       count(DISTINCT f.scene_id) AS scenes, count(DISTINCT f.h3_cell) AS occupied_cells,
       median(f.emission_rate_kg_hr) FILTER (WHERE f.review_eligible) AS median_observed_rate_kg_hr,
       count(*) FILTER (WHERE f.plume_quality='not_supplied') AS unrated_observations
FROM fact_observations f JOIN dim_sector s USING(sector_code) GROUP BY 1,2;

CREATE OR REPLACE TABLE mart_locations AS
SELECT h3_cell, count(*) AS observations, count(DISTINCT scene_id) AS scenes,
       count(DISTINCT CAST(observed_at AS DATE)) AS observation_days,
       count(DISTINCT CAST(observed_at AS DATE)) FILTER (WHERE review_eligible) AS eligible_observation_days,
       min(observed_at) AS first_observed_at, max(observed_at) AS last_observed_at,
       max(emission_rate_kg_hr) FILTER (WHERE review_eligible) AS peak_observed_rate_kg_hr,
       median(emission_rate_kg_hr) FILTER (WHERE review_eligible) AS median_observed_rate_kg_hr,
       arg_max(plume_id, emission_rate_kg_hr) FILTER (WHERE review_eligible) AS peak_plume_id,
       count(*) FILTER (WHERE plume_quality='not_supplied') AS unrated_observations,
       string_agg(DISTINCT sector_code, ', ' ORDER BY sector_code) AS sector_codes
FROM fact_observations GROUP BY h3_cell;

CREATE OR REPLACE TABLE mart_review_queue AS
SELECT row_number() OVER (ORDER BY (eligible_observation_days >= 2) DESC,
                                  peak_observed_rate_kg_hr DESC, h3_cell) AS review_order,
       *, eligible_observation_days >= 2 AS repeated_detection_cell
FROM mart_locations WHERE peak_observed_rate_kg_hr IS NOT NULL;
"""


def build_models(con):
    lookup = pd.DataFrame(SECTORS.items(), columns=["sector_code", "sector_name"])
    con.register("sector_lookup", lookup)
    con.execute(MODELS_SQL)
    con.unregister("sector_lookup")
    cells = con.execute("SELECT h3_cell FROM mart_locations ORDER BY h3_cell").fetchall()
    centers = [(c[0], *h3.cell_to_latlng(c[0])) for c in cells]
    centers = pd.DataFrame(centers, columns=["h3_cell", "center_latitude", "center_longitude"])
    con.register("centers", centers)
    con.execute("CREATE OR REPLACE TABLE dim_location AS SELECT * FROM centers")
    con.unregister("centers")


def warehouse_checks(con):
    checks = {
        "unique_observation_ids": "SELECT count(*)-count(DISTINCT observation_id) FROM fact_observations",
        "methane_only": "SELECT count(*) FROM fact_observations WHERE gas <> 'CH4'",
        "valid_coordinates": "SELECT count(*) FROM fact_observations WHERE NOT (latitude BETWEEN -90 AND 90 AND longitude BETWEEN -180 AND 180)",
        "hidden_estimates_masked": "SELECT count(*) FROM fact_observations WHERE hide_emission AND (emission_rate_kg_hr IS NOT NULL OR uncertainty_kg_hr IS NOT NULL)",
        "sector_dimension_complete": "SELECT count(*) FROM fact_observations f LEFT JOIN dim_sector s USING(sector_code) WHERE s.sector_code IS NULL",
        "monthly_counts_reconcile": "SELECT abs((SELECT count(*) FROM fact_observations)-(SELECT coalesce(sum(observations),0) FROM mart_monthly))",
        "location_counts_reconcile": "SELECT abs((SELECT count(*) FROM fact_observations)-(SELECT coalesce(sum(observations),0) FROM mart_locations))",
    }
    result = [{"check": name, "severity": "error", "failures": con.execute(sql).fetchone()[0]}
              for name, sql in checks.items()]
    if any(row["failures"] for row in result):
        raise ValueError("Warehouse reconciliation failed: " + json.dumps(result))
    return result

### 2.5. Data products: exported tables, maps, and report

The report shows coverage and a transparent review queue. All cells are exported; the browser map is capped for responsiveness. Rates describe individual observations, and quality uncertainty remains explicit.

In [ ]:
%%writefile -a methane_pipeline.py
# %% Data products: exported tables, maps, and report
def export_products(con, cfg, run_id, metrics):
    import folium
    import plotly.express as px
    from folium.plugins import MarkerCluster

    directory = cfg.work / "gold" / run_id
    directory.mkdir(parents=True, exist_ok=False)
    tables = ["fact_observations", "dim_platform", "dim_sector", "dim_date", "dim_location",
              "mart_monthly", "mart_sectors", "mart_locations", "mart_review_queue"]
    frames = {}
    for name in tables:
        df = con.execute(f"SELECT * FROM {name}").df()
        df.to_parquet(directory / f"{name}.parquet", index=False)
        df.to_csv(directory / f"{name}.csv", index=False)
        frames[name] = df
    # Partitions make date-scoped reads possible without opening the whole snapshot.
    observations = frames["fact_observations"]
    partitioned = observations.assign(year=observations.observed_at.dt.year,
                                     month=observations.observed_at.dt.month)
    partitioned.to_parquet(directory / "observations_by_date", partition_cols=["year", "month"], index=False)
    monthly = frames["mart_monthly"].sort_values("observation_month")
    sectors = frames["mart_sectors"].sort_values("observations", ascending=False)
    queue = frames["mart_review_queue"].sort_values("review_order")
    locations = frames["mart_locations"].merge(frames["dim_location"], on="h3_cell", validate="one_to_one")
    plotted = locations.sort_values(["eligible_observation_days", "peak_observed_rate_kg_hr"],
                                    ascending=[False, False], na_position="last").head(cfg.max_map_cells)
    m = folium.Map(location=[20, 0], zoom_start=2, tiles="OpenStreetMap", prefer_canvas=True)
    cluster = MarkerCluster(name="Observed grid cells").add_to(m)
    for row in plotted.to_dict("records"):
        rate = row["peak_observed_rate_kg_hr"]
        rate_text = "Unavailable" if pd.isna(rate) else f"{rate:,.0f} kg/hr"
        repeated = row["eligible_observation_days"] >= 2
        color = "#d97706" if repeated else "#087f8c"
        if pd.isna(rate):
            color = "#64748b"
        popup = (f"<b>Grid cell {html.escape(row['h3_cell'])}</b><br>"
                 f"Observations: {row['observations']}<br>Detection dates: {row['observation_days']}<br>"
                 f"Largest eligible observation: {rate_text}<br>"
                 f"Unrated observations: {row['unrated_observations']}<br>"
                 "A grid cell may contain multiple facilities. Source: Carbon Mapper.")
        folium.CircleMarker([row["center_latitude"], row["center_longitude"]], radius=5,
                            color=color, fill=True, fill_opacity=0.75,
                            popup=folium.Popup(popup, max_width=320)).add_to(cluster)
    folium.LayerControl().add_to(m)
    m.save(str(directory / "map.html"))

    fig = px.bar(monthly, x="observation_month", y="observations", color="instrument",
                 labels={"observation_month": "Observation month", "observations": "Published plume observations"},
                 title="Catalog observations over time")
    fig.update_layout(template="plotly_white", font_family="Arial", margin=dict(l=30,r=20,t=60,b=30))
    chart1 = fig.to_html(full_html=False, include_plotlyjs=True)
    fig2 = px.bar(sectors, x="sector_name", y="observations", title="Observations by provider-attributed sector",
                  labels={"sector_name": "Sector", "observations": "Observations"})
    fig2.update_layout(template="plotly_white", font_family="Arial", margin=dict(l=30,r=20,t=60,b=30))
    chart2 = fig2.to_html(full_html=False, include_plotlyjs=False)
    repeated_count = int((locations.eligible_observation_days >= 2).sum())
    summary = {**metrics, "current_observations": len(observations), "grid_cells": len(locations),
               "repeated_detection_cells": repeated_count, "mapped_cells": len(plotted),
               "earliest_observation": str(observations.observed_at.min()),
               "latest_observation": str(observations.observed_at.max()),
               "unrated_observations": int((observations.plume_quality == "not_supplied").sum()),
               "missing_rate_observations": int(observations.emission_rate_kg_hr.isna().sum()),
               "source": "Carbon Mapper", "scope": cfg.scope}
    write_json(directory / "summary.json", summary)
    view = queue[["review_order", "h3_cell", "eligible_observation_days", "peak_observed_rate_kg_hr",
                  "unrated_observations", "sector_codes"]].head(25).copy()
    view.columns = ["Order", "Grid cell", "Eligible detection dates", "Peak observed kg/hr", "Unrated observations", "Sectors"]
    table_html = view.to_html(index=False, escape=True, float_format=lambda x: f"{x:,.1f}")
    map_src = html.escape((directory / "map.html").read_text(), quote=True)
    document = f"""<!doctype html><html lang="en"><head><meta charset="utf-8">
    <meta name="viewport" content="width=device-width,initial-scale=1"><title>Satellite Methane Leak Intelligence</title>
    <style>body{{margin:0;background:#f4f7fa;color:#172b3a;font:16px/1.55 Arial,sans-serif}}
    main{{max-width:1180px;margin:auto;padding:32px}}header{{background:#103d48;color:white;padding:32px;border-radius:16px}}
    h1{{font-size:34px;margin:4px 0}}h2{{font-size:23px}}.eyebrow{{letter-spacing:.15em;font-size:12px}}
    .cards{{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:16px;margin:24px 0}}
    .card,section{{background:white;padding:22px;border-radius:12px;margin-bottom:20px}}.number{{font-size:30px;font-weight:bold}}
    .muted{{color:#526574;font-size:14px}}.note{{border-left:4px solid #d97706;padding:14px;background:#fff7e8}}
    iframe{{width:100%;height:530px;border:0}}.table-scroll{{overflow-x:auto}}table{{border-collapse:collapse;width:100%;font-size:13px}}
    th,td{{padding:10px;text-align:left;border-bottom:1px solid #e2e8f0}}th{{background:#eaf3f5}}a{{color:#087f8c}}
    </style></head><body><main><header><div class="eyebrow">ENVIRONMENTAL DATA ENGINEERING</div>
    <h1>Satellite Methane Leak Intelligence</h1><p>Published methane observations, revision history, and a spatial review queue.</p>
    <div>Source: Carbon Mapper · Input archive cutoff {html.escape(metrics['cutoff_utc'])}</div></header>
    <div class="cards"><div class="card">Observations<div class="number">{len(observations):,}</div></div>
    <div class="card">Occupied grid cells<div class="number">{len(locations):,}</div></div>
    <div class="card">Repeated detection cells<div class="number">{repeated_count:,}</div></div>
    <div class="card">Unrated observations<div class="number">{summary['unrated_observations']:,}</div></div></div>
    <p class="note">This is a screening tool for published satellite detections. A grid cell is not a verified facility.
    Rates describe individual observations; they are not annual emissions. No detection can also mean no suitable observation.</p>
    <section><h2>Locations to investigate</h2><p class="muted">Showing {len(plotted):,} of {len(locations):,} occupied H3 cells at resolution {cfg.h3_resolution}.
    Amber means detections on at least two eligible dates; teal means a single eligible date; gray has no eligible rate.
    The map needs internet for map tiles and Leaflet assets. All cells are included in the exported tables.</p>
    <iframe title="Satellite observation map" srcdoc="{map_src}"></iframe></section>
    <section>{chart1}<p class="muted">Changes in counts can reflect coverage, publication delays, and sensor deployment. These counts do not establish emissions trends.</p></section>
    <section>{chart2}</section><section><h2>Review queue</h2>
    <p>Repeated eligible detection dates come first, then the largest observed rate. Unknown quality ratings remain marked;
    questionable and bad ratings are excluded from rate-based screening. This order is a transparent triage rule.</p>
    <div class="table-scroll">{table_html}</div></section>
    <section><h2>How to interpret this report</h2><p>Observation range: {html.escape(summary['earliest_observation'])} to {html.escape(summary['latest_observation'])}.
    Missing rate values: {summary['missing_rate_observations']:,}. The full export retains uncertainty, quality flags, instrument, processing version, and raw-page references.</p>
    <p>H3 boundaries can split one site or group several sites. Distinct detection dates are not a detection-frequency estimate without the denominator of all usable overpasses.</p>
    <p>Data provided by Carbon Mapper, Inc. Data and derivatives remain subject to its
    <a href="https://carbonmapper.org/terms">non-commercial terms and attribution requirements</a>.
    Processing here includes filtering, normalization, quality masking, H3 grouping, and SQL aggregation. This project is independently developed and is not endorsed by Carbon Mapper.</p>
    <p class="muted">Run {html.escape(run_id)} · Pipeline version {CONTRACT_VERSION}</p></section></main></body></html>"""
    (directory / "report.html").write_text(document, encoding="utf-8")
    return directory, summary

### 2.6. Orchestration, update state, and failure recovery

A successful database transaction precedes watermark publication. Auto mode periodically does a full reconciliation; incremental runs overlap the prior watermark. An isolated replay demonstrates duplicate prevention.

In [ ]:
%%writefile -a methane_pipeline.py
# %% Orchestration, update state, and failure recovery
def run_pipeline(cfg, mode="auto", replay_manifest=None, session=None, make_report=True):
    if mode not in {"auto", "full", "incremental", "replay"}:
        raise ValueError("mode must be auto, full, incremental, or replay.")
    cfg.work.mkdir(parents=True, exist_ok=True)
    run_id = utcnow().strftime("%Y%m%dT%H%M%S%fZ") + "_" + uuid.uuid4().hex[:6]
    cutoff = iso(utcnow())
    log = RunLog(cfg.work / "logs" / f"{run_id}.jsonl", run_id)
    started = time.monotonic()
    with project_lock(cfg.work):
        state_path = cfg.work / "state.json"
        state = json.loads(state_path.read_text()) if state_path.exists() else {}
        if mode == "auto":
            full_due = not state or (utcnow() - datetime.fromisoformat(state["last_full_utc"].replace("Z", "+00:00"))).days >= cfg.full_refresh_days
            mode = "full" if full_due else "incremental"
        if mode == "incremental" and not state:
            raise ValueError("An incremental update needs a completed full load first. Use mode='full'.")
        lower = None
        if mode == "incremental":
            lower = iso(datetime.fromisoformat(state["watermark_utc"].replace("Z", "+00:00")) - timedelta(hours=cfg.overlap_hours))
        log.event("run_started", mode=mode, scope=cfg.scope_id)
        con = duckdb.connect(str(cfg.db_path))
        initialize_db(con)
        con.execute("INSERT INTO pipeline_runs VALUES (?, ?, CAST(? AS TIMESTAMPTZ), NULL, 'running', NULL, NULL)",
                    [run_id, mode, cutoff])
        transaction, committed = False, False
        try:
            if mode == "replay":
                if not replay_manifest:
                    raise ValueError("Replay requires the path to a completed raw manifest.")
                source_manifest = Path(replay_manifest).resolve()
                manifest, _ = read_archive(source_manifest, cfg.scope_id)
                destination = cfg.work / "bronze" / manifest["run_id"]
                if source_manifest.parent != destination.resolve():
                    if destination.exists():
                        existing = json.loads((destination / "manifest.json").read_text())
                        if existing != manifest:
                            raise ValueError("An archive with the same run ID has different content.")
                    else:
                        shutil.copytree(source_manifest.parent, destination)
                manifest_path = destination / "manifest.json"
            else:
                manifest_path = extract(cfg, run_id, cutoff, mode, lower, log, session=session)
            manifest, records = read_archive(manifest_path, cfg.scope_id)
            frame, rejected, checks = normalize(records, cfg)
            quality_dir = cfg.work / "quality" / run_id
            write_json(quality_dir / "checks.json", checks)
            write_json(quality_dir / "quarantine.json", rejected)
            if rejected:
                raise ValueError(f"{len(rejected)} observations failed critical checks. See {quality_dir / 'quarantine.json'}. "
                                 "The current warehouse and watermark have been preserved.")
            if mode == "full" and frame.empty:
                raise ValueError("Full extraction returned no observations. Check the selected scope before replacing data.")
            silver = cfg.work / "silver" / f"{run_id}.parquet"
            silver.parent.mkdir(parents=True, exist_ok=True)
            frame.to_parquet(silver, index=False)
            log.event("normalized", rows=len(frame), rejected=len(rejected))
            con.execute("BEGIN TRANSACTION")
            transaction = True
            changes = load_current(con, frame, run_id, cutoff, full_refresh=mode == "full")
            build_models(con)
            checks.extend(warehouse_checks(con))
            con.executemany("INSERT INTO quality_results VALUES (?, ?, ?, ?)",
                            [(run_id, c["check"], c["severity"], c["failures"]) for c in checks])
            write_json(quality_dir / "checks.json", checks)
            current_rows = con.execute("SELECT count(*) FROM fact_observations").fetchone()[0]
            versions = con.execute("SELECT count(*) FROM observation_versions").fetchone()[0]
            metrics = {"run_id": run_id, "mode": mode, "cutoff_utc": manifest["cutoff_utc"],
                       "run_started_at_utc": cutoff,
                       "scope_id": cfg.scope_id, "extracted_rows": len(records),
                       "accepted_rows": len(frame), "current_observations": current_rows,
                       "history_versions": versions, "raw_pages": len(manifest["pages"]),
                       "manifest": str(manifest_path.relative_to(cfg.work)),
                       "quality_checks": len(checks), "critical_failures": 0, **changes}
            if make_report:
                directory, summary = export_products(con, cfg, run_id, metrics)
            else:
                directory, summary = None, metrics
            metrics["duration_seconds"] = round(time.monotonic() - started, 3)
            summary.update(metrics)
            if directory:
                write_json(directory / "summary.json", summary)
            con.execute("UPDATE pipeline_runs SET finished_at=now(), status='succeeded', metrics_json=? WHERE run_id=?",
                        [json.dumps(metrics), run_id])
            con.execute("COMMIT")
            transaction = False
            committed = True
            # I advance the watermark only after the warehouse and data products succeed.
            if mode != "replay":
                state = {"scope_id": cfg.scope_id, "watermark_utc": cutoff,
                         "last_full_utc": cutoff if mode == "full" else state["last_full_utc"],
                         "last_successful_run": run_id}
                write_json(state_path, state)
            latest = {"run_id": run_id, "scope_id": cfg.scope_id, "manifest": str(manifest_path.relative_to(cfg.work)),
                      "products": str(directory.relative_to(cfg.work)) if directory else None,
                      "metrics": metrics}
            write_json(cfg.work / "latest.json", latest)
            log.event("run_succeeded", observations=current_rows, new=changes["new"],
                      updated=changes["updated"], removed=changes["removed"])
            return {"work": str(cfg.work), "database": str(cfg.db_path),
                    "manifest": str(manifest_path), "products": str(directory) if directory else None,
                    "metrics": metrics, "summary": summary}
        except Exception as exc:
            if transaction:
                con.execute("ROLLBACK")
            status = "committed_checkpoint_failed" if committed else "failed"
            con.execute("UPDATE pipeline_runs SET status=?, finished_at=now(), error=? WHERE run_id=?",
                        [status, f"{type(exc).__name__}: {exc}"[:4000], run_id])
            log.event(status, error_type=type(exc).__name__, message=str(exc)[:1500])
            raise
        finally:
            con.close()


def verify_replay(cfg, manifest_path, destination):
    # A separate warehouse lets me demonstrate safe reprocessing without changing the main run.
    destination = Path(destination).resolve()
    if destination in {Path(cfg.root).resolve(), cfg.work.resolve()}:
        raise ValueError("The replay check needs a separate destination from the main warehouse.")
    # A fresh folder also keeps a repeated notebook run from depending on old test artifacts.
    check_root = destination / ("check_" + uuid.uuid4().hex[:12])
    test_cfg = dataclasses.replace(cfg, root=str(check_root))
    first = run_pipeline(test_cfg, mode="replay", replay_manifest=manifest_path, make_report=False)
    with duckdb.connect(first["database"], read_only=True) as con:
        before = con.execute("SELECT observation_id, record_hash FROM fact_observations ORDER BY observation_id").fetchall()
    second = run_pipeline(test_cfg, mode="replay", replay_manifest=manifest_path, make_report=False)
    with duckdb.connect(second["database"], read_only=True) as con:
        after = con.execute("SELECT observation_id, record_hash FROM fact_observations ORDER BY observation_id").fetchall()
    passed = (before == after and second["metrics"]["new"] == 0 and second["metrics"]["updated"] == 0
              and first["metrics"]["history_versions"] == second["metrics"]["history_versions"]
              and not (test_cfg.work / "state.json").exists())
    if not passed:
        raise AssertionError("Raw replay changed the canonical observations or added duplicate versions.")
    return {"passed": True, "observations": len(after), "second_load_new": 0,
            "second_load_updated": 0, "history_versions": second["metrics"]["history_versions"],
            "watermark_created_by_replay": False}


def fetch_preview(observation_id, destination):
    # Asking for a fresh image link avoids relying on an expired signature from an older run.
    with requests.Session() as session:
        result = session.get(API, params={"plume_ids": observation_id, "limit": 1}, timeout=60)
        result.raise_for_status()
        items = result.json()["items"]
        if not items:
            raise ValueError("This observation is no longer available in the public catalog.")
        url = items[0].get("plume_png")
        if not url or urlsplit(url).scheme != "https" or urlsplit(url).hostname != "catalog.carbonmapper.org":
            raise ValueError("No supported plume image is available for this observation.")
        response = session.get(url, timeout=60)
        response.raise_for_status()
        if not response.content.startswith(b"\x89PNG\r\n\x1a\n"):
            raise ValueError("The image response was not a PNG.")
        path = Path(destination)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(response.content)
        return path


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", default=None, help="JSON containing Config fields")
    parser.add_argument("--root", default=None)
    parser.add_argument("--mode", default="auto", choices=["auto", "full", "incremental", "replay"])
    parser.add_argument("--manifest", default=None, help="Completed raw manifest for replay mode")
    args = parser.parse_args()
    values = json.loads(Path(args.config).read_text()) if args.config else {}
    if args.root:
        values["root"] = args.root
    result = run_pipeline(Config(**values), mode=args.mode, replay_manifest=args.manifest)
    print(json.dumps(result["summary"], indent=2))


if __name__ == "__main__":
    main()

## 3. Supporting files and tests

These cells write the archive helpers and offline tests. The small artificial observations are test fixtures only; they never replace live observations in the report. The documentation, SQL examples, CI workflow, and Dockerfile are included in the download for later use outside Colab.

In [ ]:
%%writefile workspace_tools.py
"""Small archive helpers for a local Colab working directory and Drive checkpoints."""
import hashlib
import os
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath


def safe_restore(archive, destination, allowed_roots=None):
    # I check every path before extracting anything into the working directory.
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as bundle:
        for info in bundle.infolist():
            name = PurePosixPath(info.filename)
            target = (destination / info.filename).resolve()
            if (name.is_absolute() or ".." in name.parts or "\\" in info.filename
                    or not target.is_relative_to(destination)
                    or stat.S_ISLNK(info.external_attr >> 16)):
                raise ValueError("Archive contains an unsafe file path.")
            if allowed_roots and (not name.parts or name.parts[0] not in allowed_roots):
                raise ValueError("This is not the expected project checkpoint.")
        corrupt = bundle.testzip()
        if corrupt:
            raise ValueError(f"Archive checksum failed for {corrupt}")
        destination.mkdir(parents=True, exist_ok=True)
        bundle.extractall(destination)


def make_archive(project, destination, roots):
    project, destination = Path(project).resolve(), Path(destination).resolve()
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp = destination.with_suffix(destination.suffix + ".partial")
    # I choose the included folders so the virtual environment and other notebooks stay out.
    ignored = {"__pycache__", ".pytest_cache", ".git", ".venv", ".ipynb_checkpoints"}
    with zipfile.ZipFile(temp, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as bundle:
        for root in roots:
            path = project / root
            paths = [path] if path.is_file() else sorted(path.rglob("*")) if path.exists() else []
            for item in paths:
                relative = item.relative_to(project)
                if (item.is_file() and not ignored.intersection(relative.parts)
                        and item.name not in {".pipeline.lock"}
                        and not item.name.endswith((".tmp", ".wal", ".partial"))
                        and item.resolve() not in {destination, temp}):
                    bundle.write(item, str(relative))
    os.replace(temp, destination)
    return destination


def copy_verified(source, destination):
    # The old checkpoint stays available until the new copy has matching bytes.
    source, destination = Path(source), Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp = destination.with_name(destination.name + ".partial")
    def sha(path):
        with path.open("rb") as f:
            return hashlib.file_digest(f, "sha256").hexdigest()
    shutil.copy2(source, temp)
    if sha(source) != sha(temp):
        temp.unlink(missing_ok=True)
        raise OSError("The checkpoint copy did not pass its checksum check.")
    os.replace(temp, destination)
    return destination

In [ ]:
%%writefile tests/test_pipeline.py
"""Small fixtures here are synthetic and only used to test failure handling."""
import copy
import gzip
import json
from pathlib import Path

import duckdb
import pandas as pd
import pytest
import requests

from methane_pipeline import (Config, RunLog, build_models, digest, extract, fetch_json,
    initialize_db, load_current, normalize, project_lock, read_archive, run_pipeline, stable_value, timestamp, verify_replay, warehouse_checks)


def observation(**updates):
    item = {
        "id": "test-observation-1", "plume_id": "test-plume-1", "gas": "CH4",
        "scene_id": "test-scene-1", "scene_timestamp": "2025-01-01T12:00:00Z",
        "modified": "2025-02-01T12:00:00Z", "published_at": "2025-01-05T00:00:00Z",
        "instrument": "tan", "platform": "Planet Tanager 1", "status": "published",
        "geometry_json": {"type": "Point", "coordinates": [-121.5, 37.4]},
        "emission_auto": 100.0, "emission_uncertainty_auto": 20.0,
        "hide_emission": False, "plume_quality": "good", "sector": "6A",
    }
    item.update(updates)
    return item


class Response:
    def __init__(self, body, status=200):
        self.body, self.status_code, self.headers = body, status, {}

    def json(self):
        return self.body

    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.HTTPError(str(self.status_code))


class Session:
    def __init__(self, responses):
        self.responses, self.calls = list(responses), []

    def get(self, url, params, timeout):
        self.calls.append(dict(params))
        return self.responses.pop(0)


def cfg(tmp_path, **kwargs):
    return Config(root=str(tmp_path), request_pause=0, **kwargs)


def frame(item, config):
    result, rejected, _ = normalize([(item, "bronze/test/page.json.gz")], config)
    assert not rejected
    return result


def db():
    con = duckdb.connect(":memory:")
    initialize_db(con)
    return con


def test_hidden_estimates_never_enter_analysis(tmp_path):
    df = frame(observation(hide_emission=True), cfg(tmp_path))
    assert df.emission_rate_kg_hr.isna().all()
    assert df.uncertainty_kg_hr.isna().all()
    assert not df.review_eligible.any()


def test_coordinates_are_longitude_then_latitude(tmp_path):
    df = frame(observation(), cfg(tmp_path))
    assert df.longitude.iloc[0] == -121.5 and df.latitude.iloc[0] == 37.4
    bad = observation(geometry_json={"coordinates": [37.4, -121.5]})
    _, rejected, _ = normalize([(bad, "raw")], cfg(tmp_path))
    assert "invalid_coordinates" in rejected[0]["reasons"]


@pytest.mark.parametrize("updates,reason", [
    ({"gas": "CO2"}, "wrong_gas"),
    ({"scene_timestamp": "not a timestamp"}, "invalid_required_timestamp"),
    ({"hide_emission": None}, "missing_visibility_flag"),
    ({"instrument": "GAO"}, "wrong_instrument"),
    ({"geometry_json": "malformed"}, "invalid_geometry_type"),
])
def test_critical_contract_failures_are_quarantined(tmp_path, updates, reason):
    df, rejected, checks = normalize([(observation(**updates), "raw")], cfg(tmp_path))
    assert df.empty and reason in rejected[0]["reasons"]
    assert checks[0]["failures"] == 1


def test_signature_rotation_is_not_a_scientific_revision():
    a = observation(plume_png="https://catalog.carbonmapper.org/p.png?c=abc&Expires=1&Signature=old")
    b = observation(plume_png="https://catalog.carbonmapper.org/p.png?c=abc&Expires=2&Signature=new")
    assert digest(stable_value(a)) == digest(stable_value(b))
    b["emission_auto"] = 150.0
    assert digest(stable_value(a)) != digest(stable_value(b))


def test_timestamps_preserve_the_instant_and_reject_invalid_inputs():
    expected = pd.Timestamp("2025-01-01T12:00:00Z")
    assert timestamp("2025-01-01T07:00:00-05:00") == expected
    assert timestamp("2025-01-01T12:00:00") == expected
    assert timestamp("not a date") is None
    assert timestamp(None) is None
    assert timestamp(1735732800) is None


def test_retry_429_then_success(tmp_path):
    config = cfg(tmp_path)
    session = Session([Response({}, 429), Response({"items": []})])
    result = fetch_json(session, {}, config, RunLog(tmp_path / "test.jsonl", "test"), sleep=lambda _: None)
    assert result == {"items": []} and len(session.calls) == 2


def test_auth_failure_does_not_retry_or_substitute_data(tmp_path):
    session = Session([Response({}, 403)])
    with pytest.raises(PermissionError):
        fetch_json(session, {}, cfg(tmp_path), RunLog(tmp_path / "log", "test"), sleep=lambda _: None)
    assert len(session.calls) == 1


def test_pagination_uses_filtered_bbox_count(tmp_path):
    config = cfg(tmp_path, page_size=1)
    a, b = observation(), observation(id="test-2", plume_id="test-plume-2")
    session = Session([Response({"items": [a], "bbox_count": 2, "total_count": 900}),
                       Response({"items": [b], "bbox_count": 2, "total_count": 900})])
    path = extract(config, "test", "2025-03-01T00:00:00Z", "full", None,
                   RunLog(tmp_path / "log", "test"), session)
    manifest, rows = read_archive(path, config.scope_id)
    assert manifest["complete"] and len(rows) == 2
    assert [p["offset"] for p in session.calls] == [0, 1]


def test_repeated_page_fails_instead_of_silently_truncating(tmp_path):
    config = cfg(tmp_path, page_size=1)
    body = {"items": [observation()], "bbox_count": 2}
    with pytest.raises(RuntimeError, match="Duplicate IDs"):
        extract(config, "test", "2025-03-01T00:00:00Z", "full", None,
                RunLog(tmp_path / "log", "test"), Session([Response(body), Response(body)]))
    assert not json.loads((config.work / "bronze/test/manifest.json").read_text())["complete"]


def test_source_count_drift_aborts_collection(tmp_path):
    config = cfg(tmp_path, page_size=1)
    session = Session([Response({"items": [observation()], "bbox_count": 2}),
                       Response({"items": [observation(id="two")], "bbox_count": 3})])
    with pytest.raises(RuntimeError, match="source changed"):
        extract(config, "test", "2025-03-01T00:00:00Z", "full", None,
                RunLog(tmp_path / "log", "test"), session)


def test_raw_archive_tampering_is_detected(tmp_path):
    config = cfg(tmp_path)
    path = extract(config, "test", "2025-03-01T00:00:00Z", "full", None,
                   RunLog(tmp_path / "log", "test"),
                   Session([Response({"items": [observation()], "bbox_count": 1})]))
    with gzip.open(path.parent / "page_00000.json.gz", "wb") as f:
        f.write(b'{"items": []}')
    with pytest.raises(ValueError, match="checksum"):
        read_archive(path)


def test_idempotent_upsert_retains_history_and_blocks_stale_replay(tmp_path):
    config, con = cfg(tmp_path), db()
    try:
        a = frame(observation(), config)
        assert load_current(con, a, "one", "2025-03-01T00:00:00Z", False)["new"] == 1
        assert load_current(con, a, "two", "2025-03-01T00:00:00Z", False)["updated"] == 0
        b = frame(observation(emission_auto=200, modified="2025-02-02T00:00:00Z"), config)
        assert load_current(con, b, "three", "2025-03-01T00:00:00Z", False)["updated"] == 1
        load_current(con, a, "four", "2025-03-01T00:00:00Z", False)
        assert con.execute("SELECT emission_rate_kg_hr FROM fact_observations").fetchone()[0] == 200
        assert con.execute("SELECT count(*) FROM observation_versions").fetchone()[0] == 2
        build_models(con)
        assert all(x["failures"] == 0 for x in warehouse_checks(con))
    finally:
        con.close()


def test_full_refresh_reconciles_withdrawals_but_delta_does_not(tmp_path):
    config, con = cfg(tmp_path), db()
    try:
        a, b = observation(), observation(id="two", plume_id="two")
        both, _, _ = normalize([(a, "raw"), (b, "raw")], config)
        load_current(con, both, "one", "2025-03-01T00:00:00Z", True)
        one = frame(a, config)
        load_current(con, one, "two", "2025-03-02T00:00:00Z", False)
        assert con.execute("SELECT count(*) FROM fact_observations").fetchone()[0] == 2
        assert load_current(con, one, "three", "2025-03-03T00:00:00Z", True)["removed"] == 1
        assert con.execute("SELECT count(*) FROM observation_versions").fetchone()[0] == 2
    finally:
        con.close()


def test_bad_batch_does_not_change_warehouse_or_watermark(tmp_path):
    config = cfg(tmp_path)
    config.work.mkdir(parents=True)
    with duckdb.connect(str(config.db_path)) as con:
        initialize_db(con)
        load_current(con, frame(observation(), config), "seed", "2025-03-01T00:00:00Z", False)
    state = {"watermark_utc": "2025-03-01T00:00:00Z", "last_full_utc": "2025-03-01T00:00:00Z"}
    (config.work / "state.json").write_text(json.dumps(state))
    session = Session([Response({"items": [observation(gas="CO2")], "bbox_count": 1})])
    with pytest.raises(ValueError, match="critical checks"):
        run_pipeline(config, mode="incremental", session=session, make_report=False)
    assert json.loads((config.work / "state.json").read_text()) == state
    with duckdb.connect(str(config.db_path), read_only=True) as con:
        assert con.execute("SELECT count(*) FROM fact_observations").fetchone()[0] == 1
        assert con.execute("SELECT status FROM pipeline_runs").fetchone()[0] == "failed"


def test_report_failure_rolls_back_new_rows_and_keeps_checkpoint(tmp_path, monkeypatch):
    import methane_pipeline
    config = cfg(tmp_path)
    config.work.mkdir(parents=True)
    state = {"watermark_utc": "2025-03-01T00:00:00Z", "last_full_utc": "2025-03-01T00:00:00Z"}
    (config.work / "state.json").write_text(json.dumps(state))
    with duckdb.connect(str(config.db_path)) as con:
        initialize_db(con)
        load_current(con, frame(observation(), config), "seed", "2025-03-01T00:00:00Z", False)
    def broken_export(*_args):
        raise OSError("simulated storage failure")
    monkeypatch.setattr(methane_pipeline, "export_products", broken_export)
    session = Session([Response({"items": [observation(id="new", plume_id="new")], "bbox_count": 1})])
    with pytest.raises(OSError, match="storage failure"):
        run_pipeline(config, mode="incremental", session=session)
    assert json.loads((config.work / "state.json").read_text()) == state
    with duckdb.connect(str(config.db_path), read_only=True) as con:
        assert con.execute("SELECT observation_id FROM fact_observations").fetchall() == [("test-observation-1",)]
        assert con.execute("SELECT count(*) FROM observation_versions").fetchone()[0] == 1


def test_empty_incremental_update_preserves_existing_data(tmp_path):
    config = cfg(tmp_path)
    run_pipeline(config, mode="full", make_report=False,
                 session=Session([Response({"items": [observation()], "bbox_count": 1})]))
    result = run_pipeline(config, mode="incremental", make_report=False,
                          session=Session([Response({"items": [], "bbox_count": 0})]))
    assert result["metrics"]["current_observations"] == 1
    assert result["metrics"]["new"] == result["metrics"]["updated"] == 0


def test_checkpoint_failure_is_distinguished_from_rolled_back_failure(tmp_path, monkeypatch):
    import methane_pipeline
    config = cfg(tmp_path)
    original = methane_pipeline.write_json
    def failing_checkpoint(path, value):
        if Path(path).name == "state.json":
            raise OSError("simulated checkpoint failure")
        return original(path, value)
    monkeypatch.setattr(methane_pipeline, "write_json", failing_checkpoint)
    with pytest.raises(OSError, match="checkpoint failure"):
        run_pipeline(config, mode="full", make_report=False,
                     session=Session([Response({"items": [observation()], "bbox_count": 1})]))
    with duckdb.connect(str(config.db_path), read_only=True) as con:
        assert con.execute("SELECT count(*) FROM fact_observations").fetchone()[0] == 1
        assert con.execute("SELECT status FROM pipeline_runs").fetchone()[0] == "committed_checkpoint_failed"
    assert not (config.work / "state.json").exists()


def test_replay_verification_can_be_run_again_in_a_fresh_isolated_folder(tmp_path):
    config = cfg(tmp_path / "main")
    manifest = extract(config, "original", "2025-03-01T00:00:00Z", "full", None,
                       RunLog(tmp_path / "log", "original"),
                       Session([Response({"items": [observation()], "bbox_count": 1})]))
    destination = tmp_path / "checks"
    assert verify_replay(config, manifest, destination)["passed"]
    assert verify_replay(config, manifest, destination)["passed"]
    assert len(list(destination.glob("check_*"))) == 2
    assert manifest.exists()
    assert not config.db_path.exists()


def test_replay_preserves_input_cutoff_separately_from_execution_time(tmp_path):
    config = cfg(tmp_path / "source")
    cutoff = "2025-03-01T00:00:00Z"
    manifest = extract(config, "original", cutoff, "full", None,
                       RunLog(tmp_path / "log", "original"),
                       Session([Response({"items": [observation()], "bbox_count": 1})]))
    target = cfg(tmp_path / "target")
    result = run_pipeline(target, mode="replay", replay_manifest=manifest, make_report=False)
    assert result["metrics"]["cutoff_utc"] == cutoff
    assert result["metrics"]["run_started_at_utc"] != cutoff


def test_lock_excludes_another_writer_and_recovers_after_an_error(tmp_path):
    (tmp_path / ".pipeline.lock").write_text("leftover file from an interrupted process")
    with pytest.raises(ValueError, match="simulated interruption"):
        with project_lock(tmp_path):
            with pytest.raises(RuntimeError, match="holds the warehouse lock"):
                with project_lock(tmp_path):
                    pytest.fail("The second writer should not acquire the lock.")
            raise ValueError("simulated interruption")
    with project_lock(tmp_path):
        pass

In [ ]:
%%writefile tests/test_workspace.py
import zipfile
from pathlib import Path

import pytest

from workspace_tools import copy_verified, make_archive, safe_restore


def test_checkpoint_round_trip_preserves_files_and_excludes_lock(tmp_path):
    project = tmp_path / "project"
    (project / "runtime/scope").mkdir(parents=True)
    (project / "runtime/scope/state.json").write_text('{"watermark": "test"}')
    (project / "runtime/scope/.pipeline.lock").write_text("stale process")
    archive = make_archive(project, tmp_path / "checkpoint.zip", ["runtime"])
    restored = tmp_path / "restored"
    safe_restore(archive, restored, allowed_roots={"runtime"})
    assert (restored / "runtime/scope/state.json").read_text() == '{"watermark": "test"}'
    assert not (restored / "runtime/scope/.pipeline.lock").exists()


@pytest.mark.parametrize("name", ["../outside.txt", "/absolute.txt", "runtime/../../outside.txt"])
def test_archive_path_traversal_is_rejected(tmp_path, name):
    archive = tmp_path / "bad.zip"
    with zipfile.ZipFile(archive, "w") as f:
        f.writestr(name, "unexpected")
    with pytest.raises(ValueError, match="unsafe"):
        safe_restore(archive, tmp_path / "destination")
    assert not (tmp_path / "destination").exists()


def test_copy_verified_replaces_previous_checkpoint(tmp_path):
    source, destination = tmp_path / "new.zip", tmp_path / "drive/latest.zip"
    destination.parent.mkdir()
    destination.write_bytes(b"older checkpoint")
    source.write_bytes(b"new checkpoint")
    copy_verified(source, destination)
    assert destination.read_bytes() == source.read_bytes()
    assert not destination.with_name("latest.zip.partial").exists()

In [ ]:
# Keeping these reference files in the export makes the project usable outside the notebook.
SUPPORT_FILES = {'requirements.txt': 'pandas>=2.2,<3\nduckdb==1.4.3\npyarrow==19.0.1\nrequests==2.32.5\nfolium==0.20.0\nplotly==6.3.0\nh3==4.3.1\npytest==8.4.2\nfilelock==3.20.0\n', 'pytest.ini': '[pytest]\npythonpath = .\ntestpaths = tests\n', '.gitignore': 'runtime/\nreplay_check/\nverification/\nexports/\n*.duckdb*\n*.zip\n*.ipynb_checkpoints/\n.venv/\n__pycache__/\n.pytest_cache/\n.env\nverified_data/\n', 'README.md': '# Satellite Methane Leak Intelligence\n\nA complete batch data engineering project for turning published satellite methane observations into an auditable warehouse and a spatial investigation queue. The pipeline answers: **where have methane plumes been observed repeatedly, which observations have larger reported rates, and how trustworthy and current is the underlying data?**\n\nSource: Carbon Mapper. This project uses published processed observations; it does not detect plumes from raw satellite pixels. It does not establish facility ownership, regulatory violations, annual emissions, or methane reductions.\n\n## Start in Google Colab\n\n1. Download `Satellite_Methane_Data_Platform.ipynb` and upload it at [Google Colab](https://colab.research.google.com/).\n2. Use a standard CPU Python runtime. No GPU is needed.\n3. Run all cells in order. The default fetches global published CH4 observations from the Tanager and EMIT satellite instruments, from 2024-01-01 to the run cutoff.\n4. The final cells display the report and offer a results ZIP. The notebook creates all its own source and support files; the accompanying project ZIP is optional.\n\n`USE_GOOGLE_DRIVE = False` is the friction-free default. Setting it to `True` adds Google\'s Drive authorization prompt and restores/saves one project checkpoint ZIP. DuckDB always runs on the local VM disk. The final checkpoint is copied to Drive only after the pipeline closes the database. A runtime that terminates before that checkpoint can lose that session\'s local work.\n\nFor a short regional run, set `BBOX = (-125, 32, -114, 42)`. This is a geographic rectangle, not an exact administrative California boundary. Changing dates, instruments, bounding box, or H3 resolution selects a different scope directory and warehouse. Default `MODE = "auto"` does a full first load, incremental updates thereafter, and a full reconciliation after seven days.\n\nThe notebook starts with empty code outputs and execution counts. Running it creates normal cell outputs, local JSONL pipeline logs, and database audit rows. Colab manages its own notebook history and service records. Sharing a notebook can share saved outputs; see [Colab\'s FAQ](https://research.google.com/colaboratory/faq.html).\n\n## Architecture\n\n```mermaid\nflowchart TD\n    A["Carbon Mapper API"] --> B["Bronze: JSON pages and checksums"]\n    B --> C["Silver: typed Parquet and quality checks"]\n    C --> D["DuckDB: current facts and revisions"]\n    D --> E["Gold: SQL marts, report and exports"]\n    D --> F["Run audit and committed watermark"]\n    F --> A\n```\n\n| Layer | Contents | Purpose |\n|---|---|---|\n| Bronze | Original compressed JSON responses, query, cutoff, counts, SHA-256 manifest | Exact input provenance and offline replay |\n| Silver | One typed Parquet batch per run; rejected IDs and reasons | Inspectable transformations and quality gating |\n| Warehouse | Current observation fact, append-only encountered revisions, dimensions, run audit, retirements | Idempotent updates and accountable corrections |\n| Gold | Monthly, sector, H3 location, and review-queue marts; CSV/Parquet; report | SQL analysis and reviewable outputs |\n| Operations | JSONL logs, quality results, watermark, scope fingerprint | Rerun safety and failure diagnosis |\n\nThe logical observation key is the provider\'s UUID `id`. The human-readable `plume_id` is retained separately. The current table stores one version per observation ID; `observation_versions` retains each distinct encountered content hash. This is encountered revision history, not a reconstruction of revisions that happened between collection runs, and not a full valid-time SCD2 model.\n\n## What makes the pipeline reliable\n\n- Pagination uses the filtered `bbox_count`, not a potentially larger `total_count`. Duplicate pages, count drift, premature empty pages, and a page-limit overrun stop ingestion.\n- HTTP 429, transient server errors, and connection timeouts retry with bounded backoff. Authentication/access failures remain explicit.\n- Every live extraction has a fixed upper modification-time cutoff. Incremental runs overlap the previous watermark by 48 hours to catch late catalog updates.\n- Replaying a completed archive verifies every page checksum. Reprocessing unchanged observations adds neither duplicate facts nor duplicate history versions.\n- Signed image URL expiration/signature parameters are excluded from content hashes, avoiding false revisions. The content identifier is preserved.\n- Older revisions cannot overwrite newer provider modification times. Equal-time changed content is accepted as the most recently encountered payload.\n- Critical data errors quarantine the batch and stop publication. Missing optional estimates or quality ratings are retained as nulls/flags, not invented values.\n- A transaction covers fact changes, history, models, quality results, and successful-run audit. A report/export failure rolls those database changes back.\n- Only a complete live full refresh reconciles observations missing from the published catalog. An incremental batch or replay never treats absence as deletion.\n- A per-scope lock prevents two writers. The watermark advances after database commit. If checkpoint publication fails after commit, the audit status is `committed_checkpoint_failed`; an idempotent rerun recovers from the older watermark.\n\nOffset pagination is not provider-side snapshot isolation. The cutoff and count/duplicate checks catch several consistency failures, but cannot prove the source never changed during collection. Periodic full reconciliation limits the remaining risk. A failed extraction starts a new run rather than resuming an old offset against a changed catalog.\n\n## Local execution\n\nPython 3.12 was used for validation. From this directory:\n\n```bash\npython -m venv .venv\nsource .venv/bin/activate\npython -m pip install -r requirements.txt\npython -m pytest -q\npython methane_pipeline.py --config config.example.json --mode auto\n```\n\nWindows uses `.venv\\Scripts\\activate` for activation. `config.example.json` uses a relative `runtime` directory. Paths are resolved from the directory in which the command runs.\n\n```bash\npython methane_pipeline.py --root runtime --mode full\npython methane_pipeline.py --root runtime --mode incremental\n```\n\nThe first command initializes the scope. The second uses its committed watermark. A zero-row delta is a successful no-change update, not an error. The report still reflects the complete current warehouse.\n\nOffline rebuild, using the supplied verified archives after extracting the project ZIP:\n\n```bash\npython replay_archives.py --source verified_data --root offline_runtime\n```\n\nThe supplied data uses the default global scope. The helper selects the latest complete full snapshot and applies all subsequent archived incremental batches in cutoff order. It verifies page checksums and rebuilds the warehouse and report into a fresh output root. Replay preserves provider observation timestamps and source cutoffs while recording the new execution time separately. It creates no live-ingestion watermark. Starting live ingestion in that root requires a full refresh.\n\nFor a single archived batch in Colab, `OFFLINE_MANIFEST` can point to a completed `verified_data/<scope_id>/bronze/<run_id>/manifest.json` after extracting the bundle. Keep the same scope settings. For the full saved dataset including incremental additions, use the rebuild command above in Colab or locally. Package installation and browser map assets still need internet.\n\nThe source bundle keeps the verified warehouse and its raw-page lineage together under `verified_data/<scope_id>/`. Only its latest gold export is packaged to avoid repeating identical report/table snapshots. `latest_report.html` is a convenient copy of that report. A new normal live run uses `runtime/`, leaving the reference snapshot separate.\n\n## Scheduling and containers\n\nThe included CLI can be scheduled on a machine or job runner you control. For example, after replacing the absolute paths in this illustrative cron entry:\n\n```cron\n0 6 * * * cd /absolute/project && /absolute/project/.venv/bin/python methane_pipeline.py --config config.example.json --mode auto >> /absolute/project/scheduler.log 2>&1\n```\n\nThe Dockerfile packages the same batch CLI for a separate container host:\n\n```bash\ndocker build -t methane-pipeline .\ndocker run --rm -v "$(pwd)/runtime:/data" methane-pipeline --root /data --mode auto\n```\n\nThe included GitHub Actions workflow runs offline tests on pushes and pull requests; it does not fetch data or schedule production runs. Neither a scheduler nor a cloud service has been deployed by this project. Colab is the interactive build/run environment, not an always-on production scheduler.\n\n## Data products\n\n`runtime/<scope_id>/latest.json` points to the latest successfully published run. Open its `gold/<run_id>/report.html` in a browser. Charts embed Plotly JavaScript; the map needs network access to Leaflet assets and map tiles. The map displays at most 2,000 cells for responsiveness and states the total number of occupied cells. All cells remain in the tables. Report text and popups use H3 cell centers; they are not facility boundaries.\n\nThe exported observation dataset is partitioned by observation year/month. Each gold run is a full current-state snapshot, even when ingestion was incremental. That is an intentional simplicity/storage tradeoff at this dataset size. Retention is manual; repeated runs accumulate raw history and exports.\n\n`sql/analyses.sql` includes read-only queries for operational health, quality availability, repeated detection cells, and the largest eligible observations. `docs/DATA_DICTIONARY.md` defines the modeled fields and table grains.\n\n## Scientific interpretation\n\nThe review rule requires a positive, visible rate and a provider rating of `good` or an explicitly marked missing rating. `questionable`, `bad`, and unrecognized ratings are excluded from rate-based screening. Missing ratings are provisional, not equivalent to good quality. The queue places cells with eligible observations on at least two distinct UTC dates first, then sorts by the largest eligible observed rate. This is a transparent screening rule, not a calibrated risk score.\n\nReported kg/hr values describe individual observations and retain their provided uncertainty. They are not added into regional inventories or multiplied into annual totals. Repeated dates do not establish continuous leakage or persistence: that would require all suitable observation opportunities, including non-detections. H3 grouping can merge nearby sites or split a single site. Changing sensor coverage and publication delays can change counts without a corresponding emissions trend.\n\nSee [Carbon Mapper\'s product guide](https://carbonmapper.org/articles/product-guide) and [API schema](https://api.carbonmapper.org/api/v1/openapi.json) for source definitions. Data and derived exports are governed by the provider\'s terms; see `DATA_NOTICE.md`.\n\n## Demonstration and CV wording\n\nThe notebook demonstrates a full load, a live incremental query, and two checksum-verified replays into a separate warehouse. `verification/` records actual test outcomes and run metrics. Figures in the supplied verification report describe that run, not a fixed future catalog size.\n\nSuggested CV wording after running and understanding the project:\n\n> Built a Python and SQL pipeline for satellite methane observations using paginated API ingestion, immutable raw archives, typed Parquet, and DuckDB. Implemented overlapping incremental loads, revision tracking, quality gates, and replay tests, then published H3-based investigation queues and an interactive report.\n\nNumbers should come from the verification output of the run being described. Do not claim production deployment, a raw-imagery detection model, real-time alerts, or measured emissions reductions.\n\n## Recovery notes\n\n| Symptom | Recovery |\n|---|---|\n| API access denied | Check current Carbon Mapper API access requirements; use verified offline replay for the supplied snapshot. |\n| Page count drift or duplicate ID across pages | Rerun the collection. The partial bronze directory is retained for diagnosis. |\n| Critical data quality failure | Inspect `quality/<run_id>/quarantine.json`; fix the contract or investigate source changes before loading. |\n| Another process holds the warehouse lock | Wait for that process to finish or stop it. The OS releases the lock when the process exits; an existing lock file alone does not block a new run. |\n| `committed_checkpoint_failed` | Fix disk/permission issues and rerun. Current facts may already contain the batch; overlap/upserts prevent duplicates. |\n| Drive checkpoint unavailable | Run locally in Colab and download the final ZIP; do not assume VM files survive a reset. |\n| Optional plume image fails | The core warehouse/report is unaffected. Fresh metadata is requested for image links; the preview cell explains a failure. |\n\nFor a larger deployment, the natural next steps are object storage for bronze/Parquet, a managed task scheduler, explicit retention, and a warehouse with transactional merge support. Those are extension points, not components claimed to be running here.\n', 'DATA_NOTICE.md': "# Data provenance and reuse\n\nSource: Carbon Mapper. Data provided by Carbon Mapper, Inc.\n\nThe source observations and all data derivatives in this project remain subject to [Carbon Mapper's Terms of Use](https://carbonmapper.org/terms), including its non-commercial purpose, attribution, and redistribution conditions. This educational environmental monitoring project does not grant a different license to those data. The terms checked for this build were dated January 13, 2026; check the current terms before a new use.\n\nThe processing applied here filters published satellite methane observations, normalizes types, masks provider-withheld estimates, records quality flags, groups coordinates into H3 cells, and aggregates SQL review tables. These derivatives are distinct from Carbon Mapper's own source-level products. The project is independently developed and is not endorsed by Carbon Mapper.\n\nSource documentation: [Product guide](https://carbonmapper.org/articles/product-guide), [API reference](https://api.carbonmapper.org/api/v1/docs), and [OpenAPI schema](https://api.carbonmapper.org/api/v1/openapi.json). The raw manifest records the query, retrieval cutoff, source URL, and page checksums for each included snapshot. Raw pages may contain expired signed public image links; the pipeline requests fresh metadata for previews.\n\nSynthetic records in `tests/` exist only to exercise software behavior. They are never substituted for observations in the pipeline or report.\n", 'replay_archives.py': '"""Rebuild a scope from its latest completed full archive and subsequent deltas."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nfrom methane_pipeline import Config, run_pipeline, write_json\n\n\ndef replay_all(source, destination, scope_id=None):\n    archives = []\n    for path in Path(source).rglob("manifest.json"):\n        item = json.loads(path.read_text())\n        if item.get("complete") and item.get("mode") in {"full", "incremental"}:\n            archives.append((item, path))\n    scopes = {item["scope_id"] for item, _ in archives if item["mode"] == "full"}\n    if scope_id is None:\n        if len(scopes) != 1:\n            raise ValueError(f"Expected one archived scope; found {sorted(scopes)}. Use --scope to select one.")\n        scope_id = next(iter(scopes))\n    full = [(item, path) for item, path in archives if item["scope_id"] == scope_id and item["mode"] == "full"]\n    if not full:\n        raise ValueError("No completed full archive matches this scope.")\n    baseline = max(full, key=lambda x: x[0]["cutoff_utc"])\n    scope = baseline[0]["scope"]\n    cfg = Config(root=str(destination), start_date=scope["start_date"], end_date=scope["end_date"],\n                 bbox=scope["bbox"], instruments=scope["instruments"], h3_resolution=scope["h3_resolution"])\n    if cfg.scope_id != scope_id:\n        raise ValueError("This archive requires a different contract version.")\n    if cfg.work.exists():\n        raise ValueError("A rebuild needs a fresh output root. Choose another --root; existing data has not been removed.")\n    # Starting at the most recent full snapshot avoids bringing back older withdrawn observations.\n    deltas = sorted([(item, path) for item, path in archives\n                     if item["scope_id"] == scope_id and item["mode"] == "incremental"\n                     and item["cutoff_utc"] > baseline[0]["cutoff_utc"]], key=lambda x: x[0]["cutoff_utc"])\n    selected = [baseline, *deltas]\n    for index, (_, path) in enumerate(selected):\n        result = run_pipeline(cfg, mode="replay", replay_manifest=path,\n                              make_report=index == len(selected) - 1)\n    write_json(cfg.work / "rebuild.json", {\n        "archives_replayed": len(selected), "scope_id": cfg.scope_id,\n        "source_run_ids": [item["run_id"] for item, _ in selected],\n        "metrics": result["metrics"],\n    })\n    return result\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--source", default="runtime", help="Directory containing archived full and incremental runs")\n    parser.add_argument("--root", default="offline_runtime", help="Fresh output root")\n    parser.add_argument("--scope", default=None, help="Scope fingerprint when the source contains multiple scopes")\n    args = parser.parse_args()\n    result = replay_all(args.source, args.root, args.scope)\n    print(json.dumps(result["summary"], indent=2))\n', 'docs/DATA_DICTIONARY.md': '# Data dictionary\n\n## Grain and relationships\n\n| Table | Grain / key | Notes |\n|---|---|---|\n| `fact_observations` | One current observation; `observation_id` primary key | Current best encountered provider revision |\n| `observation_versions` | `observation_id, record_hash` primary key | Distinct versions encountered, plus first-seen run/time |\n| `dim_platform` | Instrument/platform pair | Labels supplied by the provider |\n| `dim_sector` | `sector_code` | Unrecognized codes retained with an unmapped label |\n| `dim_date` | Observed UTC date | Dates present in the facts, not a dense calendar |\n| `dim_location` | `h3_cell` | H3 cell center latitude/longitude |\n| `mart_monthly` | Observation month/instrument | Observation, scene, occupied-cell and eligible counts |\n| `mart_sectors` | Sector code | Counts and median eligible observed rate |\n| `mart_locations` | H3 cell | Counts, distinct dates, peak/median eligible rate |\n| `mart_review_queue` | H3 cell with an eligible rate | Transparent ordering by repeated dates then peak rate |\n| `pipeline_runs` | Run ID | Mode, start/finish, status, metrics and error |\n| `quality_results` | Run/check name | Severity and number of affected records |\n| `retirements` | Run/observation ID | Missing IDs reconciled during a complete live full refresh |\n\n## Observation fields\n\n| Modeled field | Source / transformation | Meaning |\n|---|---|---|\n| `observation_id` | `id` | Provider UUID, logical record identity |\n| `plume_id` | `plume_id` | Human-readable plume name |\n| `gas` | `gas`, restricted to CH4 | Methane-only dataset |\n| `scene_id` | `scene_id` | Identity of the originating scene |\n| `observed_at` | `scene_timestamp`, UTC | Observation time, distinct from publication and processing time |\n| `modified_at` | `modified`, UTC | Provider revision time used for update ordering |\n| `published_at` | `published_at`, UTC or null | Provider publication time when supplied |\n| `instrument` | `instrument` | `tan` or `emi` in this project |\n| `platform` | `platform` | Provider platform label |\n| `sector_code` | `sector` or `unknown` | Provider attribution; not independent facility identification |\n| `longitude`, `latitude` | GeoJSON coordinates, reordered into named fields | Original point in degrees; bounds validated |\n| `emission_rate_kg_hr` | `emission_auto` | Visible, finite, non-negative instantaneous reported rate, otherwise null |\n| `uncertainty_kg_hr` | `emission_uncertainty_auto` | Provider uncertainty magnitude, otherwise null; not re-estimated here |\n| `wind_speed_m_s` | `wind_speed_avg_auto` | Optional finite source value |\n| `wind_direction_deg` | `wind_direction_avg_auto` | Optional finite source value |\n| `hide_emission` | `hide_emission` | Provider visibility flag; true masks rate and uncertainty in analysis |\n| `plume_quality` | `plume_quality` or `not_supplied` | Rating preserved separately from eligibility |\n| `review_eligible` | Rule described in README | Positive visible rate and good/explicitly missing rating |\n| `processing_version` | `emission_version` or `unknown` | Source processing version |\n| `h3_cell` | H3 of latitude/longitude, resolution 8 by default | Spatial aggregation key, not a facility ID |\n| `quality_flags` | Semicolon-separated flags | Missing, withheld, negative, or unrecognized optional values |\n| `record_hash` | SHA-256 of canonicalized source item | Ignores temporary signing parameters in HTTPS image URLs |\n| `raw_page` | Relative archive path | Traceback to an immutable input page |\n\nAll warehouse timestamps use UTC. In history, `first_seen_at` is the pipeline run time, not when the source first existed. Source `published_at` may be unavailable. Metadata added upstream is retained in bronze and participates in the canonical hash, even if it is not yet projected into a warehouse column.\n\n## Quality contract\n\nCritical rejection covers missing identities, non-CH4 gas, an out-of-scope instrument/status, malformed point geometry, out-of-bounds coordinates or requested region, missing required timestamps/scene, and a missing boolean visibility flag. Any critical rejection stops the entire batch before publication. Optional missing estimates/uncertainty/ratings raise warnings and remain inspectable. Negative or non-finite numeric estimates become null, never zero substitutes.\n\nWarehouse checks reconcile monthly and spatial observation counts, unique keys, CH4 scope, coordinate bounds, sector references, and hidden-estimate masking. The warning count is not a pass/fail claim about scientific validity. In particular, a missing provider quality rating remains an unresolved scientific limitation even if the engineering contract passes.\n', 'sql/models.sql': "-- These models are also embedded in methane_pipeline.py.\n\nCREATE OR REPLACE TABLE dim_platform AS\nSELECT DISTINCT instrument, platform FROM fact_observations;\n\nCREATE OR REPLACE TABLE dim_sector AS\nSELECT DISTINCT f.sector_code, coalesce(s.sector_name, 'Unmapped: ' || f.sector_code) AS sector_name\nFROM fact_observations f LEFT JOIN sector_lookup s USING (sector_code);\n\nCREATE OR REPLACE TABLE dim_date AS\nSELECT DISTINCT CAST(observed_at AS DATE) AS observation_date,\n       year(observed_at)::INTEGER AS year, month(observed_at)::INTEGER AS month,\n       dayofweek(observed_at)::INTEGER AS weekday\nFROM fact_observations;\n\nCREATE OR REPLACE TABLE mart_monthly AS\nSELECT date_trunc('month', observed_at)::DATE AS observation_month,\n       instrument, count(*) AS observations, count(DISTINCT scene_id) AS scenes,\n       count(DISTINCT h3_cell) AS occupied_cells,\n       count(*) FILTER (WHERE review_eligible) AS review_eligible_observations\nFROM fact_observations GROUP BY 1, 2;\n\nCREATE OR REPLACE TABLE mart_sectors AS\nSELECT f.sector_code, s.sector_name, count(*) AS observations,\n       count(DISTINCT f.scene_id) AS scenes, count(DISTINCT f.h3_cell) AS occupied_cells,\n       median(f.emission_rate_kg_hr) FILTER (WHERE f.review_eligible) AS median_observed_rate_kg_hr,\n       count(*) FILTER (WHERE f.plume_quality='not_supplied') AS unrated_observations\nFROM fact_observations f JOIN dim_sector s USING(sector_code) GROUP BY 1,2;\n\nCREATE OR REPLACE TABLE mart_locations AS\nSELECT h3_cell, count(*) AS observations, count(DISTINCT scene_id) AS scenes,\n       count(DISTINCT CAST(observed_at AS DATE)) AS observation_days,\n       count(DISTINCT CAST(observed_at AS DATE)) FILTER (WHERE review_eligible) AS eligible_observation_days,\n       min(observed_at) AS first_observed_at, max(observed_at) AS last_observed_at,\n       max(emission_rate_kg_hr) FILTER (WHERE review_eligible) AS peak_observed_rate_kg_hr,\n       median(emission_rate_kg_hr) FILTER (WHERE review_eligible) AS median_observed_rate_kg_hr,\n       arg_max(plume_id, emission_rate_kg_hr) FILTER (WHERE review_eligible) AS peak_plume_id,\n       count(*) FILTER (WHERE plume_quality='not_supplied') AS unrated_observations,\n       string_agg(DISTINCT sector_code, ', ' ORDER BY sector_code) AS sector_codes\nFROM fact_observations GROUP BY h3_cell;\n\nCREATE OR REPLACE TABLE mart_review_queue AS\nSELECT row_number() OVER (ORDER BY (eligible_observation_days >= 2) DESC,\n                                  peak_observed_rate_kg_hr DESC, h3_cell) AS review_order,\n       *, eligible_observation_days >= 2 AS repeated_detection_cell\nFROM mart_locations WHERE peak_observed_rate_kg_hr IS NOT NULL;\n", 'sql/analyses.sql': "-- Using UTC keeps these results aligned with the notebook's observation dates.\nSET TimeZone='UTC';\n\n-- Looking at the last runs helps me separate pipeline health from source measurements.\nSELECT run_id, mode, status, started_at, finished_at, metrics_json, error\nFROM pipeline_runs ORDER BY started_at DESC LIMIT 10;\n\n-- These are the warnings and checks recorded for the most recent run with checks.\nSELECT q.* FROM quality_results q\nJOIN pipeline_runs p USING (run_id)\nWHERE q.run_id = (SELECT q2.run_id FROM quality_results q2\n                 JOIN pipeline_runs p2 USING (run_id) ORDER BY p2.started_at DESC LIMIT 1)\nORDER BY severity, check_name;\n\n-- Multiple dates in one cell are a reason to investigate, not proof of continuous leakage.\nSELECT q.review_order, q.h3_cell, d.center_latitude, d.center_longitude,\n       q.observations, q.eligible_observation_days,\n       q.peak_observed_rate_kg_hr, q.unrated_observations\nFROM mart_review_queue q JOIN dim_location d USING (h3_cell)\nORDER BY review_order LIMIT 25;\n\n-- Keeping uncertainty alongside each rate makes the individual observations easier to interpret.\nSELECT plume_id, observed_at, platform, sector_code,\n       emission_rate_kg_hr, uncertainty_kg_hr, plume_quality, raw_page\nFROM fact_observations WHERE review_eligible\nORDER BY emission_rate_kg_hr DESC, plume_id LIMIT 20;\n\n-- Coverage and publication patterns can affect these counts.\nSELECT * FROM mart_monthly ORDER BY observation_month, instrument;\n\n-- Missing ratings should remain visible when describing the dataset.\nSELECT plume_quality, count(*) AS observations,\n       count(*) FILTER (WHERE emission_rate_kg_hr IS NULL) AS missing_rates,\n       count(*) FILTER (WHERE review_eligible) AS screening_eligible\nFROM fact_observations GROUP BY plume_quality ORDER BY observations DESC;\n\n-- Comparing the current fact with encountered history shows how many revisions have accumulated.\nSELECT (SELECT count(*) FROM fact_observations) AS current_observations,\n       (SELECT count(*) FROM observation_versions) AS encountered_versions,\n       (SELECT count(*) FROM retirements) AS recorded_retirements;\n", 'config.example.json': '{\n  "root": "runtime",\n  "start_date": "2024-01-01",\n  "end_date": null,\n  "bbox": null,\n  "instruments": ["tan", "emi"],\n  "page_size": 1000,\n  "overlap_hours": 48,\n  "full_refresh_days": 7,\n  "h3_resolution": 8,\n  "max_map_cells": 2000\n}\n', 'Dockerfile': 'FROM python:3.12-slim\nWORKDIR /app\nCOPY requirements.txt .\nRUN pip install --no-cache-dir -r requirements.txt\nCOPY methane_pipeline.py config.example.json ./\nENV PYTHONUNBUFFERED=1\nENTRYPOINT ["python", "methane_pipeline.py"]\nCMD ["--root", "/data", "--mode", "auto"]\n', '.dockerignore': 'runtime\nverified_data\nverification\nglobal_validation\nregional_smoke\n__pycache__\n.venv\n.git\n*.zip\n*.ipynb\n', '.github/workflows/tests.yml': "name: pipeline-tests\non: [push, pull_request, workflow_dispatch]\npermissions:\n  contents: read\njobs:\n  test:\n    runs-on: ubuntu-latest\n    steps:\n      - uses: actions/checkout@v4\n      - uses: actions/setup-python@v5\n        with:\n          python-version: '3.12'\n          cache: pip\n      - run: python -m pip install -r requirements.txt\n      - run: python -m pytest -q --junitxml=verification/tests.xml\n"}
for relative, contents in SUPPORT_FILES.items():
    destination = PROJECT_DIR / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(contents, encoding="utf-8")
print("Reference files written:", len(SUPPORT_FILES))

In [ ]:
# The tests cover duplicates, retries, revisions, rollback, checksums, and checkpoint archives.
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--junitxml=verification/tests.xml"],
    cwd=PROJECT_DIR, text=True, capture_output=True,
)
print(test_result.stdout)
if test_result.returncode:
    print(test_result.stderr)
    raise RuntimeError("The tests failed. The live pipeline has not started.")
(PROJECT_DIR / "verification/tests.txt").write_text(test_result.stdout, encoding="utf-8")
environment = {name: importlib.metadata.version(name) for name in
               ["pandas", "duckdb", "pyarrow", "requests", "folium", "plotly", "h3", "pytest", "filelock"]}
environment["python"] = sys.version.split()[0]
_ = (PROJECT_DIR / "verification/environment.json").write_text(json.dumps(environment, indent=2))

## 4. Optional Drive checkpoint and project configuration

The working database stays on the VM's local disk. If Drive is enabled, this cell restores the last checkpoint only when there is no local runtime folder. The final export cell saves the new checkpoint. A checkpoint is an archive of data and verification files, not an active database connection.

In [ ]:
import dataclasses
import importlib
from IPython.display import HTML, Image, display
import duckdb
import pandas as pd
import methane_pipeline
import workspace_tools

importlib.reload(methane_pipeline)
importlib.reload(workspace_tools)
from methane_pipeline import Config, fetch_preview, run_pipeline, verify_replay, write_json
from workspace_tools import copy_verified, make_archive, safe_restore

DRIVE_DIR = None
if USE_GOOGLE_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Drive mounting is only available in a managed Colab runtime. Set USE_GOOGLE_DRIVE=False here.")
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/Satellite_Methane_Data_Platform")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint = DRIVE_DIR / "methane_checkpoint.zip"
    if checkpoint.exists() and not (PROJECT_DIR / "runtime").exists():
        local_checkpoint = PROJECT_DIR.parent / "methane_checkpoint_restore.zip"
        copy_verified(checkpoint, local_checkpoint)
        safe_restore(local_checkpoint, PROJECT_DIR, allowed_roots={"runtime", "verification"})
        print("Restored the previous data checkpoint.")
    else:
        print("Using the current local data or starting a new warehouse.")

cfg = Config(root=str(PROJECT_DIR / "runtime"), start_date=START_DATE,
             end_date=END_DATE, bbox=BBOX)
write_json(PROJECT_DIR / "config.json", {**dataclasses.asdict(cfg), "root": "runtime"})
print("Scope:", cfg.scope_id)
print(json.dumps(cfg.scope, indent=2))

## 5. Collect, validate, load, and publish

This is the actual end-to-end run. `auto` performs the initial full load, then uses a 48-hour overlap on provider modification time until a full reconciliation is due. The watermark advances only after the warehouse and products succeed. Access failures or source contract changes are reported directly.

The output distinguishes engineering errors from scientific warnings. Missing provider quality ratings do not imply the measurements have been independently validated by this project.

In [ ]:
result = run_pipeline(cfg, mode="replay" if OFFLINE_MANIFEST else MODE,
                      replay_manifest=OFFLINE_MANIFEST)
first_result = result
write_json(PROJECT_DIR / "verification/initial_run.json", result["summary"])
print(json.dumps(result["summary"], indent=2))

# Replaying the most recent complete full archive gives the duplicate check a meaningful dataset.
full_manifests = []
for candidate in (cfg.work / "bronze").glob("*/manifest.json"):
    metadata = json.loads(candidate.read_text())
    if metadata.get("complete") and metadata.get("mode") == "full":
        full_manifests.append((metadata["cutoff_utc"], candidate))
replay_manifest = max(full_manifests, key=lambda x: x[0])[1] if full_manifests else Path(result["manifest"])

## 6. Prove that updates and replay behave correctly

The next cell runs a real incremental query, which may legitimately return no new records. It then loads one archived extraction twice into a separate verification warehouse. The second replay must add zero observations and zero revisions, and replay must not create a live-ingestion watermark.

In [ ]:
if RUN_INCREMENTAL_CHECK and not OFFLINE_MANIFEST:
    result = run_pipeline(cfg, mode="incremental")
    write_json(PROJECT_DIR / "verification/incremental_run.json", result["summary"])
    print("Incremental changes:", {k: result["metrics"][k] for k in ["new", "updated", "unchanged_or_older", "removed"]})

replay_result = verify_replay(cfg, replay_manifest, PROJECT_DIR / "replay_check")
write_json(PROJECT_DIR / "verification/replay.json", replay_result)
print("Replay check:", json.dumps(replay_result, indent=2))

## 7. SQL analysis and quality visibility

These queries read the committed warehouse. The first table summarizes quality availability, and the second shows the investigation queue. Cells with repeated eligible dates come first. A missing rating is kept as `not_supplied`, including when its observation is provisionally eligible for screening.

In [ ]:
def query(sql, parameters=None):
    with duckdb.connect(str(cfg.db_path), read_only=True) as connection:
        connection.execute("SET TimeZone='UTC'")
        return connection.execute(sql, parameters or []).df()

quality = query("""
SELECT plume_quality, count(*) AS observations,
       count(*) FILTER (WHERE emission_rate_kg_hr IS NULL) AS missing_rates,
       count(*) FILTER (WHERE review_eligible) AS screening_eligible
FROM fact_observations GROUP BY plume_quality ORDER BY observations DESC
""")
display(quality)
display(query("""
SELECT review_order, h3_cell, eligible_observation_days, peak_observed_rate_kg_hr,
       unrated_observations, sector_codes
FROM mart_review_queue ORDER BY review_order LIMIT 15
"""))
display(query("""
SELECT plume_id, observed_at, platform, emission_rate_kg_hr, uncertainty_kg_hr, plume_quality
FROM fact_observations WHERE review_eligible
ORDER BY emission_rate_kg_hr DESC, plume_id LIMIT 10
"""))

### 7.1. Pipeline operations

Observation time and pipeline execution time answer different questions. The audit table shows when the software ran; scene timestamps show when the measurements were made. The raw-page reference and archived checksums make individual rows traceable.

In [ ]:
display(query("""
SELECT run_id, mode, status, started_at, finished_at
FROM pipeline_runs ORDER BY started_at DESC LIMIT 10
"""))
display(query("SELECT check_name, severity, failures FROM quality_results WHERE run_id=? ORDER BY severity, check_name",
              [result["metrics"]["run_id"]]))

# Reading one date partition is useful when I do not need the whole observation snapshot.
products_dir = Path(result["products"])
partition_paths = sorted((products_dir / "observations_by_date").glob("year=*/month=*/*.parquet"))
if partition_paths:
    partition_example = pd.read_parquet(partition_paths[0])
    print("Example partition:", partition_paths[0].parent.relative_to(products_dir))
    print("Rows in that partition:", len(partition_example))
    display(partition_example[["plume_id", "observed_at", "emission_rate_kg_hr", "plume_quality"]].head())

## 8. Interactive report

The report combines a geographic map, observation counts, sector breakdown, and a review queue. The map shows up to 2,000 cells; all cells are in the exported tables. Map assets and tiles need internet access. If the embedded frame is restricted by the browser, `report.html` in the final ZIP can be opened directly.

In [ ]:
import html

report_path = products_dir / "report.html"
report_text = report_path.read_text(encoding="utf-8")
display(HTML('<iframe title="Methane observation report" style="width:100%;height:1300px;border:0" srcdoc="'
             + html.escape(report_text, quote=True) + '"></iframe>'))
print("Full report:", report_path)

### 8.1. One source plume preview

This optional cell retrieves one published plume image using a fresh signed link. It is a provider visualization for context, not an image-based model trained by this project. A preview failure does not undo the completed data pipeline.

In [ ]:
preview_status = {"requested": FETCH_ONE_PLUME_IMAGE, "saved": False}
if FETCH_ONE_PLUME_IMAGE and not OFFLINE_MANIFEST:
    example = query("SELECT observation_id, plume_id, observed_at FROM fact_observations WHERE review_eligible ORDER BY emission_rate_kg_hr DESC, plume_id LIMIT 1")
    if not example.empty:
        try:
            preview_path = fetch_preview(example.iloc[0]["observation_id"], PROJECT_DIR / "verification/plume_preview.png")
            print("Source: Carbon Mapper |", example.iloc[0]["plume_id"], "|", example.iloc[0]["observed_at"])
            display(Image(filename=str(preview_path), width=750))
            preview_status.update(saved=True, plume_id=str(example.iloc[0]["plume_id"]))
        except Exception as exc:
            preview_status["reason"] = type(exc).__name__
            print("The optional image is unavailable:", type(exc).__name__, "— the warehouse and report are complete.")
elif OFFLINE_MANIFEST:
    preview_status["reason"] = "Offline replay does not request an image."
write_json(PROJECT_DIR / "verification/preview.json", preview_status)

## 9. Save the project and results

The final ZIP includes the Python modules, tests, documentation, SQL, configuration, raw archives, database, data products, and verification results. It excludes the temporary replay warehouse and Python environment. A convenient `latest_report.html` is copied to its top level.

The Drive checkpoint, when enabled, stores the data and verification folders for a later notebook session. The downloaded notebook itself remains the starting point for regenerating code. If Drive is disabled, downloading the ZIP preserves this run beyond the lifetime of the Colab VM.

In [ ]:
import shutil

shutil.copy2(report_path, PROJECT_DIR / "latest_report.html")
write_json(PROJECT_DIR / "verification/latest_summary.json", result["summary"])
write_json(PROJECT_DIR / "verification/outcome.json", {
    "tests_passed": test_result.returncode == 0,
    "replay_passed": replay_result["passed"],
    "main_run_id": result["metrics"]["run_id"],
    "current_observations": result["metrics"]["current_observations"],
    "source": "Carbon Mapper",
})

if DRIVE_DIR is not None:
    checkpoint_zip = make_archive(PROJECT_DIR, PROJECT_DIR.parent / "methane_checkpoint.zip", ["runtime", "verification"])
    copy_verified(checkpoint_zip, DRIVE_DIR / "methane_checkpoint.zip")
    print("Drive checkpoint saved.")

export_roots = ["methane_pipeline.py", "workspace_tools.py", "replay_archives.py", "requirements.txt", "pytest.ini", "tests",
                "docs", "sql", "README.md", "DATA_NOTICE.md", "config.json", "config.example.json",
                "Dockerfile", ".dockerignore", ".gitignore", ".github", "runtime", "verification", "latest_report.html"]
RESULTS_ZIP = make_archive(PROJECT_DIR, PROJECT_DIR.parent / "Satellite_Methane_Run_Results.zip", export_roots)
print("Results ZIP:", RESULTS_ZIP)
print("Size:", round(RESULTS_ZIP.stat().st_size / 1024**2, 1), "MiB")
print("Completed observations:", f'{result["metrics"]["current_observations"]:,}')
print("Replay duplicate check:", replay_result["passed"])
if IN_COLAB and AUTO_DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(RESULTS_ZIP))
else:
    from IPython.display import FileLink
    display(FileLink(str(RESULTS_ZIP)))

## 10. Project explanation and portfolio notes

**What I can demonstrate:** a real source API, immutable raw archives, checksum verification, typed Parquet, transactional SQL transformations, overlapping incremental ingestion, encountered revision history, quality gating, a transparent spatial review queue, automated failure tests, and a downloadable report.

**Design choices I should be able to explain:**

- The modified-time watermark catches corrected older observations; the scene timestamp alone would miss those corrections.
- A two-day overlap is safe because observation IDs and content hashes make reprocessing idempotent.
- A full reconciliation catches withdrawals that a published-only incremental query cannot see.
- A missing quality rating stays provisional. A passed engineering check does not establish scientific accuracy.
- Grid cells avoid unsupported facility attribution, but can still combine or split real sites.
- The warehouse commits before the external watermark file. A failure between them is recovered by an idempotent rerun, not by claiming cross-system exactly-once execution.
- Gold files are complete current snapshots. Incremental ingestion does not imply partition-level incremental exports.

**Possible CV bullet, after running and understanding the project:**

> Built a Python/SQL satellite methane observation pipeline with paginated API ingestion, checksum-verified raw archives, Parquet storage, and a DuckDB warehouse; implemented incremental updates, revision tracking, quality gates, and replay tests, and published H3 investigation queues with an interactive report.

The accompanying source project includes a batch CLI, Dockerfile, and offline CI workflow. Scheduling and cloud deployment are documented extension paths; this notebook does not claim they have been deployed. No raw-imagery detection model, facility-level emissions inventory, or measured emissions reduction is claimed.

For later reference: `README.md`, `docs/DATA_DICTIONARY.md`, `sql/analyses.sql`, `verification/`, and the source [product guide](https://carbonmapper.org/articles/product-guide). Source: Carbon Mapper; see `DATA_NOTICE.md` before redistributing data or derivatives.